# 失敗の診断から学習データを決める
## SmolVLA × LIBERO-plus での追加学習と検証

ロボット操作モデル SmolVLA を LoRA で追加学習し、LIBERO-plus のシミュレータで
評価します。**周ごとに、変えるものを1つだけにして**、
失敗がどう変わるかを調べます。

| 周 | 変えるもの |
|---|---|
| 1周目 | 基準。Spatial のみ学習 |
| 2周目 | 推論時の `n_action_steps` だけ(再学習なし) |
| 3周目 | 学習(2周目で失敗した物体のデータを増やし、学習率を戻す)+ 2周目の推論設定 |
| 4周目 | 3周目と同じ学習に、**画像のぼやけの水増し**だけを足す(Section 14) |

## 実行環境
- Google Colab、GPU は T4 以上。Drive のマウントを許可する。
- 学習データ: `lerobot/libero_plus`
- 評価環境: LIBERO-plus(照明・背景・カメラ位置などを変えた摂動付きシミュレータ)
- 評価: 4 suite(spatial / object / goal / libero_10)× 3 環境 × 1 エピソード
  (`task_id` 0・4・8 は、同じ指示文の3つの環境バリエーション)

ベースは Physical AI 講座 2026 の基礎編アドバンスドで扱った SmolVLA の
LoRA 追加学習です。この実験で変更・追加した部分は、後述の各セルに記載しています。

## 📖 目次

### 準備(1〜7)
1. ランタイム確認 / 2. apt / 3. LeRobot / 4. HF 取得ヘルパ /
5. LIBERO-plus 環境 / 6. 定数と**実験プリセット**(6.1 実験用の道具 `.py` を作る) / 7. 学習データ選抜

### 進め方: 周ごとに変えるものは1つ
| 周 | 変えるもの | 実行 |
|---|---|---|
| 1 | 基準(Spatial のみ学習) | `RUN_NAME="r1_spatial"` で 8〜9 |
| 2 | **推論の `n_action_steps` だけ** | 同じモデルで 10.0〜10.2(再学習なし) |
| 3 | **学習**(2周目で失敗した5種類の物体を各20本・学習率 3e-4)+ 2周目の推論設定 | `RUN_NAME="r3_object"` で 6〜8、10.2 |
| 4 | **画像のぼやけの水増し**だけを足して学習 | `RUN_NAME="r4_blur"` で 6〜8、14 |

### 学習と評価(8〜9)
- 8.0 ハイパラ / 8.1 ベース重み DL
- 8.2 LoRA 学習 → **8.2b アダプタを Drive へ退避**(切断対策)。8.2 は `RUN["blur"]` が真なら `train_blur.py` で学習
- 8.2c 切断後の復元(**自動**。Drive にアダプタがあれば戻す)
- 8.3 マージ / 8.4 対照ベースライン / 8.5-8.6 Spatial 維持の確認
- 9 アドバンスド評価(4 suite × 3 環境 × 1 ep)。**suite 単位で再開可能**

### Object 改善実験(10)
- 10.0 診断 / 10.1 録画用パッチ / 10.2 n_action_steps を変えた Object 評価と録画
- 10.3 全ラウンドの比較表 / 10.4 周ごとの変更点と成功数

### 追加の検証(11〜15)
- 11 見直し間隔の比較(5 設定 × 30 回)
- 12 環境 A / B / C の確認(動画つき)と、条件別の成功数
- 13 ぼやけ対策①: 推論の前に画像を鮮明にする(アンシャープマスク)
- 14 ぼやけ対策②: 学習にぼやけを混ぜる(4周目 `r4_blur`)
- 15 結果のまとめ(周ごとの条件別の成功数)

### 実装上の工夫
- 実験条件を **`PRESETS` に集約**。周ごとの条件を同じノートで再現できる。
- 評価・学習の道具(`replan_sweep.py` / `eval_adaptive.py` / `train_blur.py`)は
  **6.1 のセルが `/content` に書き出す**(ファイルのアップロードは不要)。
- 切断対策: LoRA アダプタを学習直後に Drive へ。評価結果も Drive に保存する。

## 1. Colabランタイムを確認する

ColabのランタイムをGPUへ変更してから実行してください。

In [ ]:
# ==============================================================
# 【役割】実行環境の確認と作業場所の準備
#   Python / GPU の確認、Google Drive のマウント、環境変数の設定
# 【出力】DRIVE_BACKUP_DIR (Drive の保存先) / WORKDIR (作業フォルダ)
# ==============================================================
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

import torch

# --- Google Drive をマウント (成果物の永続化) --------------------
# Colab の /content/ 配下はランタイム終了で消えるので、
# 学習済みアダプタや評価結果を Drive にも保存できるようにマウントする.
# 認証ポップアップが出るのでブラウザで許可すること (初回のみ).
from google.colab import drive

drive.mount("/content/drive")

# Drive 上に成果物をコピーする場所. `MyDrive/smolvla_task/` 配下.
DRIVE_BACKUP_DIR = Path("/content/drive/MyDrive/smolvla_task")
DRIVE_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print(f"DRIVE_BACKUP_DIR: {DRIVE_BACKUP_DIR}")

# --- 作業ディレクトリ ------------------------------------------
# 実行速度重視で /content/ (ephemeral) に置く. Drive 直で作業すると
# 45 万小ファイルの展開などが遅くなる. 最終成果物だけ Drive にコピーする方針.
WORKDIR = Path("/content/workdir")
WORKDIR.mkdir(parents=True, exist_ok=True)
print(f"WORKDIR         : {WORKDIR}")

# --- Hugging Face Hub のノイズ抑制 ------------------------------
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["DIFFUSERS_VERBOSITY"] = "error"
# HF / LeRobot のキャッシュを WORKDIR 配下へ固定
os.environ["HF_HOME"] = str(WORKDIR / "hf_cache")
os.environ["HF_LEROBOT_HOME"] = str(WORKDIR / "lerobot_cache")
# 長時間学習で起こりやすい断片化 OOM を回避
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# LeRobot v0.6.0 は Python 3.12 以上が必須
if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12以上が必要です。")

# GPU 必須
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUランタイムを選択してください "
        "(Colab メニュー: ランタイム → ランタイムのタイプを変更 → GPU)"
    )

print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. システムパッケージを準備する

LeRobot、動画デコード、MuJoCoで必要になるパッケージを導入します。

In [ ]:
# ==============================================================
# 【役割】コマンド実行用の共通関数と、システムパッケージの導入
#   run_quiet / run_streaming / run_streaming_rolling を定義し、
#   ffmpeg や描画ライブラリを apt で入れる
# 【出力】上記 3 関数、導入済みの apt パッケージ
# ==============================================================


# 外部コマンドを stdout/stderr をまとめて捕捉しつつ実行する共通ヘルパ:
#   - 成功時はログを出さず notebook を汚さない
#   - 失敗時は末尾 6000 文字だけを例外に載せて可視化する
#   - stderr は stdout に統合して前後関係を保つ
def run_quiet(
    command: list[str],
    *,
    check: bool = True,
) -> subprocess.CompletedProcess:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        check=False,
    )

    if check and result.returncode != 0:
        raise RuntimeError(result.stdout[-6000:])

    return result


# 長時間コマンド用: 1 行受信するたびに print する streaming 版。
# 進捗が見えない = 無限ループに見える問題を避けるためのヘルパ。
# - prefix を付けて何のコマンドの出力か分かるようにする
# - 直近ログ 200 行をリングバッファで保持し、失敗時ダンプに使う
def run_streaming(
    command: list[str],
    *,
    prefix: str = "",
    check: bool = True,
) -> int:
    from collections import deque

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    recent: deque[str] = deque(maxlen=200)
    assert process.stdout is not None
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        print(f"{prefix}{line}", flush=True)
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# 長時間コマンド用 (pip install 等の出力が数百行になるケース):
# 直近 tail 行だけを ipywidgets の HTML box で "上書き更新" 表示する.
# タイムライン全部を表示すると notebook が肥大化するので、末尾だけ見せる.
def run_streaming_rolling(
    command: list[str],
    *,
    prefix: str = "",
    tail: int = 10,
    check: bool = True,
) -> int:
    from collections import deque
    import ipywidgets as widgets
    from IPython.display import display

    recent: deque[str] = deque(maxlen=tail)
    _widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_widget)

    process = subprocess.Popen(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    assert process.stdout is not None
    _count = 0
    for raw in process.stdout:
        line = raw.rstrip("\n")
        recent.append(line)
        _count += 1
        # 毎行更新すると UI が重いので 5 行ごとに widget を書換
        if _count % 5 == 0:
            _widget.value = (
                "<pre style='margin:0;font-size:11px;color:#666;"
                "line-height:1.3'>"
                + "\n".join(f"  {prefix}{l}" for l in recent)
                + "</pre>"
            )
    # 最終行まで反映
    _widget.value = (
        "<pre style='margin:0;font-size:11px;color:#666;line-height:1.3'>"
        + "\n".join(f"  {prefix}{l}" for l in recent)
        + "\n  ✅ done ({_count} lines total)".replace("{_count}", str(_count))
        + "</pre>"
    )
    return_code = process.wait()
    if check and return_code != 0:
        raise RuntimeError(
            "\n".join(recent) or f"command failed: {command}"
        )
    return return_code


# apt キャッシュ更新
run_quiet(["sudo", "apt-get", "update", "-qq"])
# 依存パッケージ内訳:
#   ffmpeg                : LeRobot の動画デコード
#   git / unzip           : このノートブックで clone / 展開に使用
#   libgl1 / libglib2.0-0 : OpenGL 系（MuJoCo/robosuite の描画）
#   libsm6 / libxext6     : X11 系ランタイム（描画ライブラリの依存）
#   libexpat1             : XML パーサ（MuJoCo の XML モデル読込）
#   libfontconfig1-dev    : matplotlib 用フォント設定
#   libmagickwand-dev     : Wand（ImageMagick binding）が LIBERO 経由で必要
run_quiet(
    [
        "sudo", "apt-get",
        "install",
        "-y",
        "-qq",
        "ffmpeg",
        "git",
        "unzip",
        "libgl1",
        "libglib2.0-0",
        "libsm6",
        "libxext6",
        "libexpat1",
        "libfontconfig1-dev",
        "libmagickwand-dev",
    ]
)

print("System packages ready.")

## 3. LeRobotをインストールする

LeRobot `v0.6.0`を使用します。
ColabでのLoRA学習に必要な互換性調整もこのセルで適用します。

In [ ]:
# ==============================================================
# 【役割】LeRobot v0.6.0 を取得して導入する(5〜20 分)
#   T4 向けの dtype パッチと、学習ログを減らすパッチもここで当てる
# 【出力】import できる lerobot / peft
# ==============================================================
# 手順:
#   1. 既存 install の除去（torchao も同時に。SmolVLA 非対応のため）
#   2. 固定 tag の LeRobot を shallow clone（進捗を stream）
#   3. bf16 非対応 GPU なら SmolVLM を fp16 に書換
#   4. lerobot_train.py の冗長ログと tqdm を軽微パッチで抑制
#   5. editable install（training + smolvla + peft の extras）— 出力を stream
#   6. torchao を再度アンインストール（依存で復活するケースの保険）
#   7. sys.modules と sys.path を掃除して clone した src を優先させる
LEROBOT_TAG = "v0.6.0"
LEROBOT_DIR = WORKDIR / "lerobot"
LEROBOT_SRC = LEROBOT_DIR / "src"

# --- 0. 安全ガード: torch のバージョン整合性チェック ------------
# 前回のセッションで torch を import 済みで、その後 pip install が
# torch を更新すると、C 拡張(.so)は旧版・Python 側は新版という
# 不整合状態になり "cannot import name '_EvalFrameOverride'" 等が発生する。
# 事前にディスク上と in-memory の torch バージョンを照合し、
# 差があればカーネル再起動を促す。
# torch.__version__ / importlib.metadata.version("torch") は環境によって
# '2.11.0+cu128' のように CUDA タグ (local version identifier) が付いたり
# 付かなかったりする. 両側から '+' 以降を落とした PEP 440 の public version だけで比較.
def _strip_local_version(v: str) -> str:
    return v.split("+", 1)[0]

_torch_version_in_memory = _strip_local_version(torch.__version__)
try:
    _torch_dist_version = _strip_local_version(
        importlib.metadata.version("torch")
    )
except importlib.metadata.PackageNotFoundError:
    _torch_dist_version = None

if _torch_dist_version and _torch_dist_version != _torch_version_in_memory:
    raise RuntimeError(
        f"torch mismatch: in-memory={_torch_version_in_memory} "
        f"on-disk={_torch_dist_version}. "
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 1. 既存 install を除去（再実行安全性のため） ----------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "lerobot",
        "torchao",
    ],
    check=False,
)

shutil.rmtree(LEROBOT_DIR, ignore_errors=True)

# --- 2. LeRobot を shallow clone（進捗を stream 表示） -----------
# ネットワークが遅いと数分かかるが、出力が多いので直近 10 行だけ表示
print("Cloning LeRobot...", flush=True)
run_streaming_rolling(
    [
        "git",
        "clone",
        "--progress",
        "--depth",
        "1",
        "--branch",
        LEROBOT_TAG,
        "https://github.com/huggingface/lerobot.git",
        str(LEROBOT_DIR),
    ],
    prefix="[git] ",
    tail=10,
)

# --- 3. dtype 互換性パッチ -------------------------------------
# T4/V100 系など bf16 未対応 GPU では bf16 で load すると壊れるので fp16 に置換
smolvlm_source = (
    LEROBOT_SRC
    / "lerobot"
    / "policies"
    / "smolvla"
    / "smolvlm_with_expert.py"
)

if not torch.cuda.is_bf16_supported():
    source = smolvlm_source.read_text(encoding="utf-8")
    source = source.replace(
        'torch_dtype="bfloat16",',
        'torch_dtype="float16",',
        1,
    )
    smolvlm_source.write_text(
        source,
        encoding="utf-8",
    )

# --- 4. lerobot_train.py の軽微パッチ ---------------------------
# 学習 subprocess のログを絞る:
#   - 設定 pformat の INFO を DEBUG に落とす
#   - inside_slurm() 判定に依らず tqdm を強制 disable
train_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_train.py"
)
source = train_script.read_text(encoding="utf-8")
source = source.replace(
    "logging.info(pformat(cfg.to_dict()))",
    "logging.debug(pformat(cfg.to_dict()))",
    1,
)
source = source.replace(
    "disable=inside_slurm(),",
    "disable=True,",
    1,
)
train_script.write_text(
    source,
    encoding="utf-8",
)

# --- 5. editable install（extras 込み） ------------------------
# --upgrade を付けない: torch を含む依存が既に満たされていれば
# 何も触らせない（在庫の C 拡張との不整合を避けるため）。
# 依存解決とビルドで 5〜20 分かかる. pip の出力は数百行になるので rolling 表示 (直近 10 行だけ).
print("Installing LeRobot (this may take a while)...", flush=True)
run_streaming_rolling(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "-e",
        f"{LEROBOT_DIR}[training,smolvla,peft]",
    ],
    prefix="[pip] ",
    tail=10,
)

# --- 5.5. インストール後 torch が動いていないことを再確認 --------
_torch_dist_version_after = _strip_local_version(
    importlib.metadata.version("torch")
)
if _torch_dist_version_after != _torch_version_in_memory:
    raise RuntimeError(
        f"pip install が torch を {_torch_version_in_memory} → "
        f"{_torch_dist_version_after} に変更しました。"
        "カーネルを再起動してこのセルを最初から実行し直してください。"
    )

# --- 6. torchao 再アンインストール ------------------------------
# 上の install 中に依存として復活することがあるため保険としてもう一度消す
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchao",
    ],
    check=False,
)

# --- 7. sys.modules と sys.path の掃除 --------------------------
# 前回セル実行時に読まれた古い lerobot/torchao モジュールを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "lerobot"
        or module_name.startswith("lerobot.")
        or module_name == "torchao"
        or module_name.startswith("torchao.")
    ):
        del sys.modules[module_name]

# clone した src を最優先で解決させるため sys.path を差し替える
sys.path = [
    item
    for item in sys.path
    if item not in {
        str(LEROBOT_DIR),
        str(LEROBOT_SRC),
    }
]
sys.path.insert(0, str(LEROBOT_SRC))
importlib.invalidate_caches()

# torchao が完全に消えたかチェック。残っていると SmolVLA が起動時に落ちる
try:
    importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    pass
else:
    raise RuntimeError("torchaoの削除に失敗しました。")

# 実際に import して、read 先が今 clone した src 配下であることを保証
# 初回 import は torch/CUDA 初期化で 30〜60 秒かかる場合がある
print("Importing lerobot (torch init may take up to a minute)...", flush=True)
import lerobot
import peft

if (
    LEROBOT_SRC.resolve()
    not in Path(lerobot.__file__).resolve().parents
):
    raise RuntimeError("LeRobotの読込先が正しくありません。")

print("LeRobot ready.")

## 4. 公開ファイルの取得処理を用意する

キャッシュを優先し、匿名アクセスの制限時は自動的に再試行します。

In [ ]:
# ==============================================================
# 【役割】Hugging Face からの取得を安定させる関数
#   429 エラーのときは待って再試行し、キャッシュがあれば再取得しない
# 【出力】run_hf_with_retry / cached_or_downloaded_snapshot
# ==============================================================
# 匿名アクセスは 429 (Too Many Requests) を食らいやすいので指数バックオフを
# 実装し、既存キャッシュがあれば実 DL を回避する。
import random
import time
from collections.abc import Callable
from typing import TypeVar

import httpx
from huggingface_hub import snapshot_download
from huggingface_hub.errors import (
    HfHubHTTPError,
    LocalEntryNotFoundError,
)

T = TypeVar("T")


# 429 のみ最大 6 回リトライ。それ以外のエラーは即再送出する。
def run_hf_with_retry(
    operation: Callable[[], T],
) -> T:
    last_error: BaseException | None = None

    for attempt in range(6):
        try:
            return operation()
        except (
            HfHubHTTPError,
            httpx.HTTPStatusError,
        ) as error:
            last_error = error
            response = getattr(error, "response", None)
            status = getattr(response, "status_code", None)

            # 429 以外はリトライしない（404 などを叩き続けない）
            if status != 429 and "429" not in str(error):
                raise

            if attempt == 5:
                break

            # サーバが Retry-After を返せば尊重、無ければ指数バックオフ
            headers = getattr(response, "headers", {}) or {}
            try:
                delay = float(
                    headers.get("Retry-After", 15)
                ) + 1
            except (TypeError, ValueError):
                delay = min(
                    120,
                    15 * (2**attempt) + random.random(),
                )

            time.sleep(delay)

    raise RuntimeError(
        "Hugging Faceからの取得に失敗しました。"
    ) from last_error


# まず local_files_only でキャッシュ確認、無ければ実 DL する。
# 大きなモデル/データセットを何度も再取得しないための最適化。
def cached_or_downloaded_snapshot(
    repo_id: str,
    revision: str,
    *,
    allow_patterns: list[str] | None = None,
    ignore_patterns: list[str] | None = None,
) -> Path:
    try:
        # キャッシュヒット時はネットワークに触れず即返す
        return Path(
            snapshot_download(
                repo_id=repo_id,
                revision=revision,
                token=False,
                allow_patterns=allow_patterns,
                ignore_patterns=ignore_patterns,
                local_files_only=True,
            )
        )
    except (
        LocalEntryNotFoundError,
        FileNotFoundError,
    ):
        # キャッシュミス → 429 リトライ付きで実 DL
        # max_workers=1 でサーバに優しくアクセス（429 回避）
        return Path(
            run_hf_with_retry(
                lambda: snapshot_download(
                    repo_id=repo_id,
                    revision=revision,
                    token=False,
                    allow_patterns=allow_patterns,
                    ignore_patterns=ignore_patterns,
                    max_workers=1,
                )
            )
        )

## 5. LIBERO-plus 評価環境を準備する

シミュレータ(MuJoCo / robosuite / LIBERO-plus)を導入し、評価用アセットを展開します。
アセットは約 6GB あります。Drive に tar.gz のキャッシュがあれば、それを展開するだけで
短時間で終わります。

In [ ]:
# ==============================================================
# 【役割】評価用シミュレータ (LIBERO-plus) を使える状態にする
#   依存ライブラリの導入、LIBERO-plus の取得、アセット (約 6GB) の展開、
#   lerobot_eval.py への進捗表示パッチ。Drive にキャッシュがあれば短縮される
# 【出力】LIBERO_PLUS_DIR / LIBERO_PLUS_ASSETS_DIR
# ==============================================================
# 手順:
#   1. MuJoCo/robosuite 系の依存を pin 付きで install
#   2. LIBERO-plus fork を固定 SHA で clone/checkout
#   3. assets.zip を HF から取得して展開
#   4. ~/.libero/config.yaml を書いてアセット位置を教える
#   5. lerobot_eval.py に進捗表示用のパッチを当てる
#   6. sys.path / sys.modules を掃除して fork 側を優先ロード
from huggingface_hub import hf_hub_download
import time

LIBERO_PLUS_SHA = "4976dc3"
LIBERO_PLUS_DIR = WORKDIR / "LIBERO-plus"
LIBERO_PLUS_PACKAGE_ROOT = (
    LIBERO_PLUS_DIR / "libero" / "libero"
)
# assets (45 万小ファイル 6GB) の保存先. /content/ 配下なので展開は高速.
LIBERO_PLUS_ASSETS_DIR = WORKDIR / "libero_plus_assets" / "assets"

# Colab のヘッドレス GPU 環境では EGL バックエンドで MuJoCo を描画
os.environ["MUJOCO_GL"] = "egl"

# --- 1. 既存 install を除去（再実行安全性） --------------------
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "hf-libero",
        "libero",
        "robosuite",
    ],
    check=False,
)

# --- 2. LIBERO-plus 依存の pin 付き install --------------------
# バージョンを揃えないと robosuite / mujoco の API 差でクラッシュするため厳格に固定
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "robosuite==1.4.1",  # LIBERO-plus が期待する固定版
        "bddl==1.0.1",
        "easydict==1.13",
        "mujoco==3.7.0",
        "matplotlib==3.10.8",
        "Wand==0.6.13",
        "scikit-image==0.25.2",
        "gym==0.26.2",
        "future",  # bddl 1.0.1 が依存宣言していない隠れ依存
    ]
)

# robosuite が期待どおりの版でロードされることを担保
if (
    importlib.metadata.version("robosuite")
    != "1.4.1"
):
    raise RuntimeError(
        "robosuite 1.4.1 is required."
    )

# --- 3. LIBERO-plus fork を clone -----------------------------
if not (LIBERO_PLUS_DIR / ".git").is_dir():
    shutil.rmtree(
        LIBERO_PLUS_DIR,
        ignore_errors=True,
    )
    run_quiet(
        [
            "git",
            "clone",
            "--quiet",
            "https://github.com/sylvestf/LIBERO-plus.git",
            str(LIBERO_PLUS_DIR),
        ]
    )

# 固定 SHA へ checkout。既存 clone に対しても再実行安全になるよう、
# checkout 失敗時のみ fetch でリカバリする
checkout = run_quiet(
    [
        "git",
        "-C",
        str(LIBERO_PLUS_DIR),
        "checkout",
        "--quiet",
        LIBERO_PLUS_SHA,
    ],
    check=False,
)

if checkout.returncode != 0:
    # SHA がローカルに無い（shallow clone 等）なら fetch してから checkout
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "fetch",
            "--quiet",
            "--depth",
            "1",
            "origin",
            LIBERO_PLUS_SHA,
        ]
    )
    run_quiet(
        [
            "git",
            "-C",
            str(LIBERO_PLUS_DIR),
            "checkout",
            "--quiet",
            LIBERO_PLUS_SHA,
        ]
    )

# fork を editable install。--no-deps で pin 済みの依存を上書きされないようにする
run_quiet(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-deps",
        "-e",
        str(LIBERO_PLUS_DIR),
    ]
)

# --- 4. assets.zip を HF から DL・展開 -------------------------
# 大きな asset は fork 側に含まれないので Sylvest/LIBERO-plus から取得
if not LIBERO_PLUS_ASSETS_DIR.is_dir():
    assets_root = WORKDIR / "libero_plus_assets"
    assets_root.mkdir(parents=True, exist_ok=True)

    # ---- Drive キャッシュ戦略 ----
    # 優先度: (1) tar.gz キャッシュ (最速)  > (2) zip キャッシュ  > (3) HF DL
    # tar.gz を Drive に持っておくと、次回セッションから展開まで含めて数分で終わる.
    _drive_tar = DRIVE_BACKUP_DIR / "libero_plus_assets.tar.gz"
    _drive_zip = DRIVE_BACKUP_DIR / "libero_plus_assets.zip"

    if _drive_tar.is_file():
        # ---- 最速パス: tar.gz を Drive から取ってきて展開 ----
        print(
            f"Drive の tar.gz キャッシュから復元中 (unzip 不要): {_drive_tar}",
            flush=True,
        )
        _t0 = time.time()
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar

        # tar.gz は zip と違って中央ディレクトリを持たないため、
        # 通常はストリーム走査しないとファイル総数が分からない.
        # そこで前回の tar 作成時に .count サイドカーへ件数を書き出しておき、
        # あればそれを使って tqdm の total を即座に埋める.
        _drive_tar_count = _drive_tar.with_suffix(_drive_tar.suffix + ".count")
        _tar_total = None
        if _drive_tar_count.is_file():
            try:
                _tar_total = int(_drive_tar_count.read_text().strip())
                print(f"サイドカーから件数を取得: {_tar_total} files", flush=True)
            except Exception:
                _tar_total = None
        if _tar_total is None:
            # サイドカー無し (旧キャッシュ): getmembers() で 1 度走査する.
            # data は展開しないのでフル展開よりは速いが、それでも数十秒待つ.
            print("件数を数えています (初回のみ、少し時間かかる)...", flush=True)
            with tarfile.open(_drive_tar, "r:gz") as _tf:
                _tar_total = sum(1 for _ in _tf)

        with tarfile.open(_drive_tar, "r:gz") as _tf:
            _pbar_tx = _tqdm_tar(total=_tar_total, desc="tar 展開", unit="files")
            while True:
                _m = _tf.next()
                if _m is None:
                    break
                _tf.extract(_m, assets_root)
                _pbar_tx.update(1)
            _pbar_tx.close()
        print(f"tar 展開完了 (in {time.time()-_t0:.0f}s)", flush=True)

        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"古い zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
        # → LIBERO_PLUS_ASSETS_DIR が存在するはず. 以下 DL/unzip はスキップ.
        _skip_dl_unzip = True
    else:
        _skip_dl_unzip = False

if not LIBERO_PLUS_ASSETS_DIR.is_dir() and not _skip_dl_unzip:
    # ---- 中速パス: zip キャッシュから復元 or HF DL ----
    local_zip = assets_root / "assets.zip"

    if _drive_zip.is_file() and not local_zip.is_file():
        print(f"Drive キャッシュから assets.zip を復元: {_drive_zip}", flush=True)
        shutil.copy2(_drive_zip, local_zip)

    if local_zip.is_file():
        archive_path = local_zip
        print(f"既存の assets.zip を使用: {archive_path}", flush=True)
    else:
        print("HF から assets.zip をダウンロード中 (6GB, 時間かかる)...", flush=True)
        # ---- HF DL 中は progress bar を有効化 ----
        # env-var の pop は huggingface_hub が import 時に disabled をキャッシュするため効かない.
        # 公式 API enable_progress_bars() / disable_progress_bars() で明示的に切り替える.
        from huggingface_hub.utils import (
            enable_progress_bars,
            disable_progress_bars,
        )
        enable_progress_bars()
        try:
            _t0 = time.time()
            archive_path = Path(
                run_hf_with_retry(
                    lambda: hf_hub_download(
                        repo_id="Sylvest/LIBERO-plus",
                        repo_type="dataset",
                        filename="assets.zip",
                        local_dir=assets_root,
                        token=False,
                    )
                )
            )
            _dl_sec = time.time() - _t0
        finally:
            disable_progress_bars()

        _size_mb = archive_path.stat().st_size / 1e6
        print(
            f"HF DL 完了: {_size_mb:.0f} MB in {_dl_sec:.0f}s "
            f"({_size_mb / max(_dl_sec, 1):.1f} MB/s)",
            flush=True,
        )
        # 注: この段階では Drive に zip を書かない.
        # 展開後に tar.gz を作って Drive に置く (下の方の処理) 方が
        # 次セッションからの復元も速くて経済的.
    extract_dir = assets_root / "extract"

    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)

    # 展開は Python の zipfile + tqdm progress bar で行う.
    # ・全体進捗は tqdm バーで見せる (ETA / files/sec 含む).
    # ・直近 10 ファイル名を ipywidgets の HTML box で "上書き更新" する
    #   (subprocess の unzip -o だと 45 万行の出力になって notebook が固まるため).
    import zipfile
    import ipywidgets as widgets
    from IPython.display import display
    from tqdm.auto import tqdm
    from collections import deque

    print(f"Extracting {archive_path.name} to {extract_dir}...", flush=True)

    # 直近展開ファイル 10 件を表示するウィジェット
    _recent_paths: deque[str] = deque(maxlen=10)
    _recent_widget = widgets.HTML(
        value=(
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>(starting...)</pre>"
        ),
        layout=widgets.Layout(margin="4px 0px 4px 32px"),
    )
    display(_recent_widget)

    _t0 = time.time()
    with zipfile.ZipFile(archive_path) as _zf:
        _members = _zf.infolist()
        _pbar = tqdm(_members, desc="unzip", unit="files")
        for _i, _m in enumerate(_pbar):
            _recent_paths.append(_m.filename)
            _zf.extract(_m, extract_dir)
            # 500 ファイルごとに widget を上書き (毎回だと重い)
            if _i % 500 == 0:
                _recent_widget.value = (
                    "<pre style='margin:0;font-size:11px;color:#666;"
                    "line-height:1.3'>"
                    + "\n".join(f"  {p}" for p in _recent_paths)
                    + "</pre>"
                )
        # 最終状態
        _recent_widget.value = (
            "<pre style='margin:0;font-size:11px;color:#666;"
            "line-height:1.3'>"
            + f"  ✅ done: {len(_members)} files extracted"
            + "</pre>"
        )
    print(f"unzip 完了 (in {time.time()-_t0:.0f}s)", flush=True)

    # zip 内のディレクトリ階層は変わりうるので rglob で assets/ を探す。
    # ネストが浅いものを優先（本物の assets ルートに近いと想定）
    candidates = sorted(
        [
            path
            for path in extract_dir.rglob("assets")
            if path.is_dir()
        ],
        key=lambda path: len(path.parts),
    )

    if not candidates:
        raise FileNotFoundError(
            "LIBERO-plus assets not found."
        )

    LIBERO_PLUS_ASSETS_DIR.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    # 展開した本物の assets を fork 内の期待位置へ移動
    shutil.move(
        str(candidates[0]),
        str(LIBERO_PLUS_ASSETS_DIR),
    )
    # assets の移動が終わったので extract テンポラリだけ削除。
    shutil.rmtree(extract_dir, ignore_errors=True)

    # ---- 次セッション用に tar.gz を Drive にキャッシュ ----
    # zip + unzip のフルパスは重いので、展開後の assets を tar.gz 化して Drive に保存.
    # 次回セッションでは tar 展開だけになり、数分で復元できる.
    try:
        _t0 = time.time()
        print(
            f"Drive に tar.gz キャッシュを作成中...",
            flush=True,
        )
        import tarfile
        from tqdm.auto import tqdm as _tqdm_tar_c

        # 作成時は事前にファイル数をカウントできるので tqdm の total を渡せる
        _entries = list(LIBERO_PLUS_ASSETS_DIR.rglob("*"))
        _total_entries = len(_entries) + 1  # +1: root dir 自身
        _pbar_tc = _tqdm_tar_c(total=_total_entries, desc="tar 作成", unit="files")

        def _tar_filter(tarinfo):
            _pbar_tc.update(1)
            return tarinfo

        with tarfile.open(_drive_tar, "w:gz", compresslevel=1) as _tf:
            _tf.add(LIBERO_PLUS_ASSETS_DIR, arcname="assets", filter=_tar_filter)
        _pbar_tc.close()
        # 次回復元時に tqdm の total を即座に出せるよう、件数サイドカーを保存.
        _drive_tar.with_suffix(_drive_tar.suffix + ".count").write_text(
            str(_total_entries) + "\n"
        )
        print(f"tar.gz キャッシュ完了 (in {time.time()-_t0:.0f}s)", flush=True)
        # 古い zip キャッシュを削除 (Drive 容量節約)
        if _drive_zip.is_file():
            try:
                _drive_zip.unlink()
                print(f"旧 zip キャッシュを削除: {_drive_zip}", flush=True)
            except Exception:
                pass
    except Exception as _e:
        print(f"tar.gz キャッシュ作成失敗 (無視して継続): {_e}", flush=True)

# --- 5. ~/.libero/config.yaml を書く --------------------------
# LIBERO は起動時にこのファイルを読んで assets/bddl/datasets/init_files の
# 場所を解決する。fork 側のパスに向ける
# ---- LIBERO の期待する assets パスへの symlink を作る ----
# LIBERO の robosuite Arena XML 読込は fork 内の相対パス
# ({LIBERO_PLUS_DIR}/libero/libero/assets/...) をハードコードで参照する.
# 一方で本 notebook は 45 万小ファイルの実体を LIBERO_PLUS_ASSETS_DIR に置いてある.
# 両者を symlink で繋ぐことで、LIBERO は "期待の場所" から実体を辿れる.
_libero_expected_assets = LIBERO_PLUS_DIR / "libero" / "libero" / "assets"
if _libero_expected_assets.is_symlink() or _libero_expected_assets.exists():
    if _libero_expected_assets.is_symlink():
        _libero_expected_assets.unlink()
    else:
        shutil.rmtree(_libero_expected_assets, ignore_errors=True)
_libero_expected_assets.parent.mkdir(parents=True, exist_ok=True)
_libero_expected_assets.symlink_to(LIBERO_PLUS_ASSETS_DIR)
print(f"symlink: {_libero_expected_assets} -> {LIBERO_PLUS_ASSETS_DIR}")

libero_config_dir = Path.home() / ".libero"
libero_config_dir.mkdir(
    parents=True,
    exist_ok=True,
)
(libero_config_dir / "config.yaml").write_text(
    "\n".join(
        [
            f"assets: {LIBERO_PLUS_ASSETS_DIR}",
            (
                "bddl_files: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'bddl_files'}"
            ),
            (
                "datasets: "
                f"{LIBERO_PLUS_PACKAGE_ROOT.parent / 'datasets'}"
            ),
            (
                "init_states: "
                f"{LIBERO_PLUS_PACKAGE_ROOT / 'init_files'}"
            ),
        ]
    )
    + "\n",
    encoding="utf-8",
)

# --- 6. lerobot_eval.py に進捗表示パッチを当てる -----------------
# デフォルトの評価スクリプトは冗長な log と tqdm を吐くので抑制。
# 代わりに "EVAL_PROGRESS task=x/y episode=a/b" 形式を stdout に流し、
# こちらのノート側でパースして表示する。
eval_script = (
    LEROBOT_SRC
    / "lerobot"
    / "scripts"
    / "lerobot_eval.py"
)
source = eval_script.read_text(encoding="utf-8")

# 冗長な設定 pformat を DEBUG に落とす
source = source.replace(
    "logging.info(pformat(asdict(cfg)))",
    "logging.debug(pformat(asdict(cfg)))",
    1,
)
# 録画時以外に rollout 動画を最大 10 本描画してしまう箇所を 0 に固定
source = source.replace(
    "max_episodes_rendered = 0 if cfg.eval.recording else 10",
    "max_episodes_rendered = 0",
    1,
)
# tqdm を強制無効化
source = source.replace(
    "disable=inside_slurm()",
    "disable=True",
)

# 現在の task/episode 進捗を保持するグローバル辞書をスクリプトへ注入
progress_state = (
    '_EVAL_PROGRESS = {"task_index": 0, "task_total": 0}'
)
if progress_state not in source:
    import_anchor = "from tqdm import trange\n"
    if import_anchor not in source:
        raise RuntimeError(
            "Evaluation progress import anchor not found."
        )
    source = source.replace(
        import_anchor,
        import_anchor + "\n" + progress_state + "\n",
        1,
    )

# task ループの先頭で _EVAL_PROGRESS を更新するコードを差し込む
task_loop_anchor = (
    "        for i, (task_group, task_id, env) "
    "in enumerate(tasks):\n"
)
task_loop_patch = (
    task_loop_anchor
    + '            _EVAL_PROGRESS["task_index"] = i + 1\n'
    + '            _EVAL_PROGRESS["task_total"] = len(tasks)\n'
)
if (
    '_EVAL_PROGRESS["task_index"] = i + 1'
    not in source
):
    if task_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation task-loop anchor not found."
        )
    source = source.replace(
        task_loop_anchor,
        task_loop_patch,
        1,
    )

# episode ループの各周で進捗 1 行を stdout に流す
episode_loop_anchor = "    for batch_ix in progbar:\n"
episode_progress_line = (
    '        print('
    'f"EVAL_PROGRESS '
    "task={_EVAL_PROGRESS['task_index']}/"
    "{_EVAL_PROGRESS['task_total']} "
    'episode={batch_ix + 1}/{n_batches}", '
    "flush=True)\n"
)
if "EVAL_PROGRESS task=" not in source:
    if episode_loop_anchor not in source:
        raise RuntimeError(
            "Evaluation episode-loop anchor not found."
        )
    source = source.replace(
        episode_loop_anchor,
        episode_loop_anchor + episode_progress_line,
        1,
    )

eval_script.write_text(
    source,
    encoding="utf-8",
)

# --- 7. sys.path 掃除と fork のロード検証 ---------------------
# pip install した LIBERO-plus fork が絶対に手前に来るようにする
libero_plus_path = str(LIBERO_PLUS_DIR)
sys.path = [
    item
    for item in sys.path
    if item != libero_plus_path
]
sys.path.insert(0, libero_plus_path)

# 古い LIBERO/robosuite モジュールキャッシュを追い出す
for module_name in list(sys.modules):
    if (
        module_name == "libero"
        or module_name.startswith("libero.")
        or module_name == "robosuite"
        or module_name.startswith("robosuite.")
    ):
        del sys.modules[module_name]

importlib.invalidate_caches()

import libero
from libero.libero import benchmark

# libero パッケージの読込先が fork ディレクトリ配下であることを保証
search_paths = [
    Path(path).resolve()
    for path in getattr(libero, "__path__", [])
]

if not any(
    LIBERO_PLUS_DIR.resolve() in path.parents
    or path == LIBERO_PLUS_DIR.resolve()
    for path in search_paths
):
    raise RuntimeError(
        "LIBERO-plus fork was not loaded."
    )

# benchmark モジュールについても同様に確認
benchmark_path = Path(
    benchmark.__file__
).resolve()

if (
    LIBERO_PLUS_DIR.resolve()
    not in benchmark_path.parents
):
    raise RuntimeError(
        "LIBERO-plus benchmark was not loaded."
    )

print("LIBERO-plus ready.")

## 6. 条件を設定する(実験プリセット)

このセルで、実験に関わる値をまとめて決めます。

- モデルとデータセットの repo・revision(再現のため固定)
- Spatial / Object のタスク名リスト
- **実験プリセット `PRESETS`** と、実行する周 `RUN_NAME`
- 出力先と Drive の退避先

In [ ]:
# ==============================================================
# 【役割】実験条件の定義。条件を変えるのはこのセルだけ
#   モデル・データの固定 revision、タスク名、実験プリセット PRESETS、
#   実行する周 RUN_NAME、出力先、Drive の退避先
# 【出力】RUN / TRAIN_SUITES / OUTPUT_DIR / MERGED_MODEL_DIR など
# ==============================================================

# --- モデル / データセットの固定 revision -----------------------
# revision を commit hash で固定して再現性を担保
BASE_MODEL_REPO = "lerobot/smolvla_libero_plus"
BASE_MODEL_REVISION = (
    "7bb70aa5bc92b82c9239142775d3a173103567ff"
)

# SmolVLA の Vision-Language backbone（500M パラメータ版）
VLM_REPO = (
    "HuggingFaceTB/SmolVLM2-500M-Video-Instruct"
)

DATASET_REPO = "lerobot/libero_plus"
DATASET_REVISION = (
    "f3f49f426d75030177b18778374005bc12ccd588"
)

# --- LIBERO-Spatial の 10 タスク（学習・評価で共通） -------------
# データセット側の task 名と厳密一致させる必要がある
SPATIAL_TASK_NAMES = [
    "pick up the black bowl from table center and place it on the plate",
    "pick up the black bowl next to the cookie box and place it on the plate",
    "pick up the black bowl next to the plate and place it on the plate",
    "pick up the black bowl next to the ramekin and place it on the plate",
    "pick up the black bowl on the cookie box and place it on the plate",
    "pick up the black bowl on the ramekin and place it on the plate",
    "pick up the black bowl on the stove and place it on the plate",
    "pick up the black bowl on the wooden cabinet and place it on the plate",
    "pick up the black bowl in the top drawer of the wooden cabinet and place it on the plate",
    "pick up the black bowl between the plate and the ramekin and place it on the plate",
]

# --- 乱数 ------------------------------------------------------
SEED = 42                     # 乱数 seed
EVAL_SEED = 2026              # 評価 seed

# --- LIBERO-Object の 10 タスク -------------------------------
OBJECT_TASK_NAMES = [
    "pick up the alphabet soup and place it in the basket",
    "pick up the bbq sauce and place it in the basket",
    "pick up the butter and place it in the basket",
    "pick up the chocolate pudding and place it in the basket",
    "pick up the cream cheese and place it in the basket",
    "pick up the ketchup and place it in the basket",
    "pick up the milk and place it in the basket",
    "pick up the orange juice and place it in the basket",
    "pick up the salad dressing and place it in the basket",
    "pick up the tomato sauce and place it in the basket",
]
SUITE_TASK_NAMES = {
    "libero_spatial": SPATIAL_TASK_NAMES,
    "libero_object": OBJECT_TASK_NAMES,
}

# ==============================================================
# 実験プリセット: ここだけ書き換えれば条件が変わる
# ==============================================================
# 3周で1つの物語にする。1周ごとに変えるものを1つに絞る。
#  1周目 r1_spatial : 基準。Spatial のみ学習。Object が全滅する
#  2周目            : 学習はしない。推論の n_action_steps だけ変える
#                     (Section 10.2 で r1_spatial のモデルを評価)
#  3周目 r3_object  : 学習を変える。2周目で失敗した5種類の物体のデータを
#                     各20本に増やし(他の物体は各5本)、学習率は3e-4に戻す
#                     + 2周目で良かった n_action_steps
#                     (データと学習率を同時に変えるので、効いた方は切り分けない)
#  4周目 r4_blur    : 3周目と同じ設定で、学習画像のメインカメラに「ぼやけ」
#                     (モーションブラー / ガウスぼかし)を確率 0.5 で混ぜる
#                     (変えるのは、ぼやけの水増しだけ。Section 14)
# r1b_spatial_object: 中間試行(参考)。Object を各5本足しただけで効果なし。
#                     loss がほぼ平らだった → 3周目で学習率を戻す根拠
_LR_R1 = dict(lr=1e-4, final_lr=1e-5, warmup=200, batch=2,
              lora_r=32, lora_alpha=32)
# 2周目 (n_action_steps=10) でも失敗した物体:
#  bbq sauce / chocolate pudding / cream cheese / ketchup / orange juice
_FAILED_IN_R2 = [OBJECT_TASK_NAMES[i] for i in (1, 3, 4, 5, 7)]
PRESETS = {
    "r1_spatial": dict(suites=["libero_spatial"], eps=5,
                       steps=5000, **_LR_R1),
    "r1b_spatial_object": dict(
        suites=["libero_spatial", "libero_object"], eps=5,
        steps=6000, **_LR_R1),
    "r3_object": dict(
        suites=["libero_object"], eps=5,
        eps_by_task={n: 20 for n in _FAILED_IN_R2},   # 失敗した物体は20本
        steps=4000,
        lr=3e-4, final_lr=3e-5, warmup=100, batch=2,
        lora_r=32, lora_alpha=32),
}
RUN_NAME = "r3_object"   # ← 1・2周目は r1_spatial、3周目は r3_object、4周目は r4_blur
# 4周目: 3周目と同じ設定 + ぼやけの水増し
PRESETS["r4_blur"] = dict(PRESETS["r3_object"], blur=True, blur_p=0.5)
RUN = PRESETS[RUN_NAME]
TRAIN_SUITES = RUN["suites"]
TRAIN_EPISODES_PER_TASK = RUN["eps"]

# --- 出力パス --------------------------------------------------
OUTPUT_DIR = WORKDIR / "outputs" / f"smolvla_{RUN_NAME}_lora"
MERGED_MODEL_DIR = WORKDIR / f"smolvla_{RUN_NAME}_merged"
BASELINE_MODEL_DIR = WORKDIR / "smolvla_libero_plus_baseline"

BASE_EVAL_DIR = WORKDIR / "eval" / "base"
FINETUNED_EVAL_DIR = WORKDIR / "eval" / f"{RUN_NAME}_spatial"
COMPARISON_CSV_PATH = WORKDIR / f"spatial_check_{RUN_NAME}.csv"
FT_COL = f"{RUN_NAME} (%)"

# --- Drive 退避先(切断対策) -----------------------------------
DRIVE_RUN_DIR = DRIVE_BACKUP_DIR / "runs" / RUN_NAME
DRIVE_EVAL_DIR = DRIVE_BACKUP_DIR / "eval"
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_EVAL_DIR.mkdir(parents=True, exist_ok=True)

# --- 評価時のカメラ対応(LIBERO 内部名 → SmolVLA の入力名)-----
EVAL_CAMERA_MAPPING = {
    "agentview_image": "front",           # 俯瞰カメラ
    "robot0_eye_in_hand_image": "wrist",  # ハンドカメラ
}

# --- 混合精度モード --------------------------------------------
# bf16 対応 GPU（A100/H100/L40/RTX40 系など）なら bf16、それ以外は fp16
MIXED_PRECISION = (
    "bf16"
    if torch.cuda.is_bf16_supported()
    else "fp16"
)

print(f"RUN_NAME={RUN_NAME}  suites={TRAIN_SUITES}  " \
      f"eps/task={TRAIN_EPISODES_PER_TASK}  steps={RUN['steps']}")

### 6.1 実験用の道具(.py)を作る

次の3つを `/content` に書き出します(**アップロードは不要**)。ランタイムが切れて
`/content` が消えたときも、このセルをもう一度実行すれば戻ります。

| ファイル | 役割 |
|---|---|
| `replan_sweep.py` | 設定を変えながら評価を実行し、成功数・推論時間を集める |
| `eval_adaptive.py` | `lerobot-eval` の前に差し込む。見直し間隔の制御・計測と、**推論前のぼやけ補正** |
| `train_blur.py` | `lerobot-train` の前に差し込む。**学習画像にぼやけを混ぜる** |

同じ内容は GitHub の `tools/` にあります(ライブラリ本体は書き換えません)。

In [ ]:
# 【役割】評価・学習で使う3つの道具(.py)を /content に作る(アップロード不要)
import pathlib, py_compile
FILES = {}
FILES['replan_sweep.py'] = r'''
import contextlib
import io
import json
import os
import shutil
import subprocess
import time
from pathlib import Path

# 比べる設定: name, モード, n_action_steps (chunk_size=50 以下), K_OPEN, K_HOLD
SETTINGS = [
    dict(name="fixed50", mode="fixed", nas=50),
    dict(name="fixed25", mode="fixed", nas=25),
    dict(name="fixed10", mode="fixed", nas=10),
    dict(name="fixed5", mode="fixed", nas=5),
    dict(name="adaptive_10_25", mode="grip", nas=50, k_open=10, k_hold=25),
]


def object_variant_ids(names, per_obj=3):
    """各物体の指示文に一致する task_id のうち、等間隔に per_obj 個を選ぶ。"""
    from libero.libero import benchmark
    with contextlib.redirect_stdout(io.StringIO()):
        b = benchmark.get_benchmark_dict()["libero_object"]()
    ids = {n: [] for n in names}
    for i in range(b.get_num_tasks()):
        lang = b.get_task(i).language.lower()
        for n in names:
            if lang.startswith(n):
                ids[n].append(i)
    out = {}
    for n, xs in ids.items():
        pos = [round(j * (len(xs) - 1) / (per_obj - 1))
               for j in range(per_obj)]
        out[n] = [xs[p] for p in dict.fromkeys(pos)]
    return out


def run_one(model_dir, st, task_ids, out_root, tools_dir, lerobot_dir,
            lerobot_src, libero_plus_dir, camera_mapping, seed,
            episode_length=None):
    out = Path(out_root) / st["name"]
    shutil.rmtree(out, ignore_errors=True)
    stats = out.parent / f"{st['name']}_stats.json"
    env = os.environ.copy()
    env.update(MUJOCO_GL="egl", TQDM_DISABLE="1", PYTHONWARNINGS="ignore",
               REPLAN_MODE=st["mode"], STATS_JSON=str(stats),
               K_OPEN=str(st.get("k_open", 10)),
               K_HOLD=str(st.get("k_hold", 25)),
               PRE_FILTER=st.get("pre", "none"),
               PRE_SIGMA=str(st.get("pre_sigma", 2.0)),
               PRE_AMOUNT=str(st.get("pre_amount", 1.5)),
               PYTHONPATH=os.pathsep.join(
                   [str(libero_plus_dir), str(lerobot_src),
                    env.get("PYTHONPATH", "")]))
    cmd = [
        "python", str(Path(tools_dir) / "eval_adaptive.py"),
        f"--policy.path={model_dir}", "--policy.device=cuda",
        "--policy.use_amp=false", f"--policy.n_action_steps={st['nas']}",
        "--env.type=libero", "--env.is_libero_plus=true",
        "--env.task=libero_object",
        "--env.task_ids=" + json.dumps(task_ids, separators=(",", ":")),
        "--env.camera_name_mapping="
        + json.dumps(camera_mapping, separators=(",", ":")),
        "--env.observation_height=256", "--env.observation_width=256",
        "--env.control_mode=relative", "--env.max_parallel_tasks=1",
        "--eval.batch_size=1", "--eval.n_episodes=1",
        "--eval.use_async_envs=false", "--eval.recording=false",
        f"--seed={seed}", f"--output_dir={out}",
    ]
    if episode_length:
        cmd.append(f"--env.episode_length={episode_length}")
    t0 = time.time()
    r = subprocess.run(cmd, cwd=lerobot_dir, env=env, capture_output=True,
                       text=True)
    wall = time.time() - t0
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        return None
    info = json.loads(next(out.rglob("eval_info.json")).read_text())
    ok = {t["task_id"]: bool(t["metrics"]["successes"][0])
          for t in info["per_task"]}
    eps = json.loads(stats.read_text())["episodes"]
    ms = [x for e in eps for x in e["call_ms"]]
    return {
        "setting": st["name"], "n": len(ok), "success": sum(ok.values()),
        "per_task": ok,
        "calls_per_episode": sum(e["calls"] for e in eps) / len(eps),
        "steps_per_episode": sum(e["steps"] for e in eps) / len(eps),
        "ms_per_call": sum(ms) / max(len(ms), 1),
        "wall_s": round(wall, 1),
    }


def run_sweep(model_dir, settings, ids_by_object, out_root, save_json,
              **kw):
    """設定ごとに全 task_id を評価。済んだ設定は save_json から読んで飛ばす。"""
    save_json = Path(save_json)
    done = json.loads(save_json.read_text()) if save_json.is_file() else []
    seen = {r["setting"] for r in done}
    task_ids = [i for xs in ids_by_object.values() for i in xs]
    for st in settings:
        if st["name"] in seen:
            print("skip", st["name"])
            continue
        print("run ", st["name"], f"({len(task_ids)} rollouts)", flush=True)
        row = run_one(model_dir, st, task_ids, out_root, **kw)
        if row is None:
            break
        done.append(row)
        save_json.write_text(json.dumps(done, indent=1))
        print(f"  success {row['success']}/{row['n']}  "
              f"calls/ep {row['calls_per_episode']:.1f}  "
              f"{row['ms_per_call']:.0f} ms/call", flush=True)
    return done


def summarize(rows):
    """設定ごとの成功率・推論回数・1回の時間・1エピソードの推論時間を表にする。"""
    import pandas as pd
    d = pd.DataFrame([{
        "設定": r["setting"],
        "成功": f"{r['success']}/{r['n']}",
        "成功率": round(r["success"] / r["n"], 2),
        "推論回数/エピソード": round(r["calls_per_episode"], 1),
        "ms/回": round(r["ms_per_call"]),
        "推論時間/エピソード(s)": round(
            r["calls_per_episode"] * r["ms_per_call"] / 1000, 1),
        "動作1回あたりの余裕(ms)": None,
    } for r in rows])
    return d.drop(columns=["動作1回あたりの余裕(ms)"])
'''
FILES['eval_adaptive.py'] = r'''
import atexit
import json
import os
import sys
import time
from pathlib import Path

MODE = os.environ.get("REPLAN_MODE", "fixed")
K_OPEN = int(os.environ.get("K_OPEN", "10"))    # つかむ前の見直し間隔
K_HOLD = int(os.environ.get("K_HOLD", "25"))    # つかんでいる間の見直し間隔
MIN_DROP = float(os.environ.get("MIN_DROP", "0.3"))  # 閉じたとみなす最小変化
STATS_JSON = os.environ.get("STATS_JSON", "")
PRE_FILTER = os.environ.get("PRE_FILTER", "none")
PRE_SIGMA = float(os.environ.get("PRE_SIGMA", "2.0"))
PRE_AMOUNT = float(os.environ.get("PRE_AMOUNT", "1.5"))
PRE_KEYS = os.environ.get("PRE_KEYS", "front,image,camera1").split(",")
PRE_MARK = os.environ.get("PRE_MARK", "")


def unsharp(x, sigma, amount):
    """x: (B, C, H, W) の画像 (0〜1)。x + amount * (x - ガウスぼかし(x))。"""
    import torch
    import torch.nn.functional as F
    r = max(1, int(3 * sigma))
    k = torch.arange(-r, r + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(-(k ** 2) / (2 * sigma ** 2))
    k = k / k.sum()
    c = x.shape[1]
    kh = k.view(1, 1, 1, -1).repeat(c, 1, 1, 1)
    kv = k.view(1, 1, -1, 1).repeat(c, 1, 1, 1)
    y = F.pad(x, (r, r, 0, 0), mode="reflect")
    y = F.conv2d(y, kh, groups=c)
    y = F.pad(y, (0, 0, r, r), mode="reflect")
    y = F.conv2d(y, kv, groups=c)
    return (x + amount * (x - y)).clamp(0.0, 1.0)


class Replanner:
    """グリッパーの開き幅 w の履歴から、いつ見直すかを決める (torch 非依存)。"""

    def __init__(self):
        self.w0 = None
        self.wmin = None
        self.since = 0

    def holding(self, w: float) -> bool:
        """初期の開き w0 から、最小値までの半分以上閉じたら『つかんでいる』。"""
        if self.w0 is None:
            self.w0 = self.wmin = w
        self.wmin = min(self.wmin, w)
        drop = self.w0 - self.wmin
        return drop > MIN_DROP and (self.w0 - w) > 0.5 * drop

    def want_replan(self, w: float) -> bool:
        k = K_HOLD if self.holding(w) else K_OPEN
        return self.since >= k


def install():
    import torch
    from lerobot.policies.smolvla import modeling_smolvla as M
    from lerobot.utils.constants import ACTION, OBS_STATE

    P = M.SmolVLAPolicy
    orig_select, orig_chunk, orig_reset = (P.select_action,
                                           P._get_action_chunk, P.reset)
    episodes, cur = [], {}

    def new_cur():
        return {"steps": 0, "calls": 0, "call_ms": [], "w": []}

    def reset(self):
        if cur.get("steps"):
            episodes.append(dict(cur))
        cur.clear()
        cur.update(new_cur())
        self._rp = Replanner()
        return orig_reset(self)

    def chunk(self, *a, **k):
        torch.cuda.synchronize()
        t = time.perf_counter()
        out = orig_chunk(self, *a, **k)
        torch.cuda.synchronize()
        cur["call_ms"].append((time.perf_counter() - t) * 1e3)
        cur["calls"] += 1
        self._rp.since = 0
        return out

    def select(self, batch, noise=None, **kw):
        s = batch[OBS_STATE].reshape(-1, batch[OBS_STATE].shape[-1])[-1]
        w = float(s[6] - s[7])           # 2 本の指の開き (正規化された値)
        cur["w"].append(round(w, 3))
        cur["steps"] += 1
        if MODE == "grip" and self._rp.want_replan(w) \
                and len(self._queues[ACTION]) > 0:
            self._queues[ACTION].clear()  # 残りの行動を捨て、次で再計画
        else:
            self._rp.holding(w)           # 状態の更新だけ
        if PRE_FILTER == "unsharp":
            for key in list(batch.keys()):
                if (key.startswith("observation.images.")
                        and key.split(".")[-1] in PRE_KEYS):
                    batch[key] = unsharp(batch[key], PRE_SIGMA, PRE_AMOUNT)
                    if PRE_MARK and not Path(PRE_MARK).exists():
                        Path(PRE_MARK).write_text(key)   # かかった印
        out = orig_select(self, batch, noise, **kw)
        self._rp.since += 1
        return out

    P.reset, P._get_action_chunk, P.select_action = reset, chunk, select

    def dump():
        if cur.get("steps"):
            episodes.append(dict(cur))
        if STATS_JSON:
            Path(STATS_JSON).write_text(json.dumps(
                {"mode": MODE, "k_open": K_OPEN, "k_hold": K_HOLD,
                 "episodes": episodes}))

    atexit.register(dump)


if __name__ == "__main__":
    install()
    from lerobot.scripts.lerobot_eval import main
    sys.argv[0] = "lerobot-eval"
    main()
'''
FILES['train_blur.py'] = r'''
import math
import os
import random
import sys

import torch
import torch.nn.functional as F

P_BLUR = float(os.environ.get("BLUR_P", "0.5"))
KEYS = set(os.environ.get("BLUR_KEYS", "front,image,camera1").split(","))
STATE = {"seen": 0, "blurred": 0, "keys": set()}


def gauss_kernel(sigma):
    r = max(1, int(3 * sigma))
    t = torch.arange(-r, r + 1, dtype=torch.float32)
    k = torch.exp(-(t ** 2) / (2 * sigma ** 2))
    k = k / k.sum()
    return (k[:, None] * k[None, :])


def motion_kernel(length, angle):
    n = length if length % 2 == 1 else length + 1
    k = torch.zeros(n, n)
    c = (n - 1) / 2
    for i in range(4 * n):
        t = -c + (2 * c) * i / (4 * n - 1)
        x = int(round(c + t * math.cos(angle)))
        y = int(round(c + t * math.sin(angle)))
        k[y, x] = 1.0
    return k / k.sum()


def random_blur_kernel():
    if random.random() < 0.7:     # モーションブラー(長さ5〜15, 向きはランダム)
        return motion_kernel(random.choice([5, 7, 9, 11, 13, 15]),
                             random.uniform(0, math.pi))
    return gauss_kernel(random.uniform(1.0, 3.0))   # ガウスぼかし


def apply_blur(x):
    """x: (..., C, H, W)。float(0〜1) か uint8。"""
    shape, dt = x.shape, x.dtype
    y = x.reshape(-1, *shape[-3:]).float()
    if dt == torch.uint8:
        y = y / 255.0
    k = random_blur_kernel()
    c = y.shape[1]
    r = k.shape[0] // 2
    w = k.to(y.device).view(1, 1, *k.shape).repeat(c, 1, 1, 1)
    y = F.conv2d(F.pad(y, (r, r, r, r), mode="reflect"), w, groups=c)
    if dt == torch.uint8:
        y = (y * 255.0).round().clamp(0, 255)
    return y.to(dt).reshape(shape)


def install():
    from lerobot.datasets.lerobot_dataset import LeRobotDataset
    orig = LeRobotDataset.__getitem__

    def getitem(self, idx):
        item = orig(self, idx)
        STATE["seen"] += 1
        cams = sorted(k for k in item if k.startswith("observation.images."))
        if random.random() < P_BLUR:
            for key in cams:
                if key.split(".")[-1] in KEYS and torch.is_tensor(item[key]):
                    item[key] = apply_blur(item[key])
                    STATE["keys"].add(key)
                    STATE["blurred"] += 1
        n = STATE["seen"]
        if n in (1, 200, 2000):
            msg = (f"seen={n} blurred={STATE['blurred']} "
                   f"blurred_keys={sorted(STATE['keys'])} cameras={cams}")
            print("[train_blur]", msg, flush=True)
            with open("/content/train_blur_log.txt", "a") as f:
                f.write(msg + "\n")
            if n == 200 and STATE["blurred"] == 0:
                raise RuntimeError("train_blur: ぼやけがかかっていません。 " + msg)
        return item

    LeRobotDataset.__getitem__ = getitem


if __name__ == "__main__":
    install()
    from lerobot.scripts.lerobot_train import main
    sys.argv[0] = "lerobot-train"
    main()
'''
for _n, _s in FILES.items():
    _p = pathlib.Path('/content') / _n
    _p.write_text(_s.lstrip('\n'), encoding='utf-8')
    py_compile.compile(str(_p), doraise=True)
    print('ok', _n, _p.stat().st_size, 'bytes')

## 7. 学習データを選ぶ

`TRAIN_SUITES` の各タスクから `TRAIN_EPISODES_PER_TASK` 本を等間隔に選びます。

In [ ]:
# ==============================================================
# 【役割】学習に使うエピソードを選ぶ
#   TRAIN_SUITES の各タスクから、等間隔に N 本を選ぶ(再現できる選び方)
# 【出力】EPISODE_INDICES (学習に使うエピソード番号)
# ==============================================================
# 学習用サブセットを再現可能に構築する。
# データセット側の task 名は表記揺れがありうるので正規化キーで突合し、
# 各 task の episode index から等間隔に 5 個を間引く（合計 50 episode）。
import re
from collections import defaultdict

from lerobot.datasets.dataset_metadata import (
    LeRobotDatasetMetadata,
)


# 小文字化・記号除去・空白正規化で突合用キーを作る。
def normalize_task_name(value: str) -> str:
    value = value.lower().replace("_", " ")
    value = re.sub(r"[^a-z0-9 ]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


# メタデータの tasks 列は str/list どちらもあり得るので単一 str に落とす。
def task_name_from_cell(value) -> str:
    if isinstance(value, str):
        return value

    try:
        if len(value) > 0:
            return str(value[0])
    except TypeError:
        pass

    return str(value)


# episode 群から等間隔に count 個を選ぶ（先頭と末尾を必ず含む）。
def choose_evenly_spaced(
    episode_indices: list[int],
    count: int,
) -> list[int]:
    if len(episode_indices) < count:
        raise ValueError(
            f"Task には episode が {len(episode_indices)} 件しかないため、"
            f"要求数 {count} を満たせません。"
        )
    positions = [
        round(
            index
            * (len(episode_indices) - 1)
            / (count - 1)
        )
        for index in range(count)
    ]

    return [
        episode_indices[position]
        for position in positions
    ]


# データセットのメタデータを取得（429 リトライ付き）
dataset_metadata = run_hf_with_retry(
    lambda: LeRobotDatasetMetadata(
        DATASET_REPO,
        revision=DATASET_REVISION,
    )
)

# 「task 名 → その task に属する episode index リスト」を構築
task_to_episodes: dict[str, list[int]] = defaultdict(list)

for episode_index, task_cell in enumerate(
    dataset_metadata.episodes["tasks"]
):
    task_to_episodes[
        task_name_from_cell(task_cell)
    ].append(int(episode_index))

# 正規化キーから実際のタスク名（表記揺れ含む）を引くための辞書
available_by_normalized = {
    normalize_task_name(task_name): task_name
    for task_name in task_to_episodes
}

# TRAIN_SUITES の各タスクから episode を選ぶ
selected_by_task: dict[str, list[int]] = {}

for suite_name in TRAIN_SUITES:
    for task_name in SUITE_TASK_NAMES[suite_name]:
        actual_task = available_by_normalized.get(
            normalize_task_name(task_name)
        )

        if actual_task is None:
            raise RuntimeError(
                f"{suite_name} task not found: {task_name}"
            )

        # 物体ごとに本数を変えたいとき用 (PRESETS の eps_by_task)
        n_eps = RUN.get("eps_by_task", {}).get(
            task_name, TRAIN_EPISODES_PER_TASK)
        selected_by_task[actual_task] = choose_evenly_spaced(
            task_to_episodes[actual_task],
            n_eps,
        )
        print(
            f"{suite_name:15s} 全{len(task_to_episodes[actual_task]):3d}本"
            f" → {n_eps:2d}本  {actual_task[:55]}"
        )

# フラットな episode index のソート済みリストへ集約
EPISODE_INDICES = sorted(
    episode_index
    for episode_indices in selected_by_task.values()
    for episode_index in episode_indices
)

expected = sum(
    RUN.get("eps_by_task", {}).get(n, TRAIN_EPISODES_PER_TASK)
    for s in TRAIN_SUITES for n in SUITE_TASK_NAMES[s]
)
if len(EPISODE_INDICES) != expected:
    raise RuntimeError("Episode selection failed.")

print(f"Training data: {len(selected_by_task)} tasks, "
      f"{len(EPISODE_INDICES)} episodes")

## 8. LoRA で追加学習する

公開の事前学習済みモデル `lerobot/smolvla_libero_plus` に、LoRA で追加学習します。
LoRA は、元の重みを凍結したまま、小さな差分だけを学習する方法です。

流れ: 初期重みの取得(8.1) → 学習(8.2) → アダプタを Drive へ退避(8.2b) →
ベース重みへマージ(8.3) → 比較用ベースラインの保存(8.4) →
Spatial の維持確認(8.5〜8.6)。

ここで作る `MERGED_MODEL_DIR`(追加学習後)と `BASELINE_MODEL_DIR`(追加学習前)が、
Section 9 の評価の入力になります。

### 8.0 ハイパラ

値は Section 6 の `PRESETS` から読み出します。変えたいときは、ここではなく
`PRESETS` を書き換えます。

| 変数 | 意味 | 効くもの |
|---|---|---|
| `STEPS` | 学習の総ステップ数 | 学習量 |
| `LEARNING_RATE` / `FINAL_LEARNING_RATE` | 学習率の最初と最後(cosine で下げる) | 1歩の大きさ。小さすぎると loss が動かない |
| `WARMUP_STEPS` | 学習率を上げていく区間 | 学習の安定 |
| `BATCH_SIZE` | 1回に使うサンプル数 | 学習の安定・VRAM |
| `LORA_R` / `LORA_ALPHA` | LoRA の大きさ | 表現力 |

変えてはいけないもの: `train_expert_only`(VLM を凍結する設定)、画像サイズ
(評価環境の 256×256 と揃える)。モデルの保存は必ず `save_pretrained` を経由させる。

In [ ]:
# ==============================================================
# 【役割】学習ハイパラを PRESETS から読み出して確認表示する
#   値を変えたいときは、ここではなく Section 6 の PRESETS を書き換える
# 【出力】STEPS / LEARNING_RATE / LORA_R など
# ==============================================================
# 値を変えたいときは、ここではなく Section 6 の PRESETS を書き換える。
STEPS               = RUN["steps"]       # 学習総 step 数
LOG_FREQ            = 100                # loss print 頻度
BATCH_SIZE          = RUN["batch"]       # ミニバッチサイズ
LEARNING_RATE       = RUN["lr"]          # peak learning rate
FINAL_LEARNING_RATE = RUN["final_lr"]    # cosine decay 後の lr
WARMUP_STEPS        = RUN["warmup"]      # linear warmup step 数
LORA_R              = RUN["lora_r"]      # LoRA 低ランク次元
LORA_ALPHA          = RUN["lora_alpha"]  # LoRA スケーリング

# ---- Spatial 維持の確認(8.5)に使う評価 ----
EVAL_TASK_IDS          = [0, 4, 8]   # 3 task × 1 ep × 2 model = 6 rollout
EVAL_EPISODES_PER_TASK = 1

print(f"[8.0] run={RUN_NAME}")
print(f"  learning: STEPS={STEPS}, LR={LEARNING_RATE}->{FINAL_LEARNING_RATE},"
      f" WARMUP={WARMUP_STEPS}, BATCH={BATCH_SIZE}, SEED={SEED}")
print(f"  LoRA    : r={LORA_R}, alpha={LORA_ALPHA}")
print(f"  data    : {len(EPISODE_INDICES)} episodes")
print(f"  eval    : TASK_IDS={EVAL_TASK_IDS}, SEED={EVAL_SEED}")

### 8.1 事前学習済み SmolVLA を DL

In [ ]:
# ==============================================================
# 【役割】追加学習の出発点になる公開モデルを取得する
# 【出力】BASE_MODEL_LOCAL (モデルの保存場所)
# ==============================================================
# 必要最小限のファイルのみ取得する。README や eval/**（動画ファイル）は落とさない。
BASE_MODEL_LOCAL = cached_or_downloaded_snapshot(
    BASE_MODEL_REPO,
    BASE_MODEL_REVISION,
    allow_patterns=[
        "config.json",                        # モデル構成
        "model.safetensors",                  # 本体 weight
        "train_config.json",                  # 学習時設定（再利用のため）
        "policy_preprocessor.json",           # 入力前処理設定
        "policy_preprocessor*.safetensors",   # 入力側 stats
        "policy_postprocessor.json",          # 出力後処理設定
        "policy_postprocessor*.safetensors",  # 出力側 stats
    ],
    ignore_patterns=[
        "README.md",
        "eval/**",  # 公開 repo に含まれる評価動画（重い）は不要
    ],
)

# 本体 weight が確実に取れていることを確認
if not (
    BASE_MODEL_LOCAL / "model.safetensors"
).is_file():
    raise FileNotFoundError("Base model not found.")

print("Base model ready.")

### 8.2 LeRobot subprocess で LoRA 学習

100 stepごとに平均lossとlearning rateを表示します。

In [ ]:
# ==============================================================
# 【役割】LoRA で追加学習する (VLM は凍結し、action expert だけ学習)
#   lerobot-train を起動し、ログを step / loss / lr の 1 行に整えて表示
# 【出力】OUTPUT_DIR/checkpoints/<STEPS>/pretrained_model (LoRA アダプタ)
# ==============================================================
# lerobot-train の CLI をそのまま呼んで LoRA 学習を実行。
# stdout をパースして step/loss/lr を LOG_FREQ ごとに 1 行だけ表示する。
import re
from collections import deque

# EPISODE_INDICES を CLI 引数として渡すため JSON 配列文字列に整形
episodes_json = (
    "["
    + ",".join(map(str, EPISODE_INDICES))
    + "]"
)

# --- lerobot-train の CLI 引数 ---------------------------------
command = [
    # 4周目(RUN["blur"])は、ぼやけの水増しを足した train_blur.py で学習する
    *(["python", "/content/train_blur.py"] if RUN.get("blur")
      else ["lerobot-train"]),
    # 初期重みと VLM backbone のパス
    f"--policy.path={BASE_MODEL_LOCAL}",
    f"--policy.vlm_model_name={VLM_REPO}",
    # Hub への push は行わない
    "--policy.push_to_hub=false",
    "--policy.repo_id=null",
    # 入出力 features はチェックポイント側の値を尊重
    "--policy.input_features=null",
    "--policy.output_features=null",
    "--policy.empty_cameras=0",
    # LoRA 対象は expert 側のみ。VLM 側は凍結する
    "--policy.freeze_vision_encoder=true",
    "--policy.train_expert_only=true",
    # 学習率と cosine スケジューラ
    f"--policy.optimizer_lr={LEARNING_RATE}",
    f"--policy.scheduler_decay_lr={FINAL_LEARNING_RATE}",
    f"--policy.scheduler_warmup_steps={WARMUP_STEPS}",
    f"--policy.scheduler_decay_steps={STEPS}",
    # データセットと選抜済み episode
    f"--dataset.repo_id={DATASET_REPO}",
    f"--dataset.revision={DATASET_REVISION}",
    f"--dataset.episodes={episodes_json}",
    "--dataset.use_imagenet_stats=false",
    "--dataset.video_backend=torchcodec",
    # 出力ディレクトリ / ジョブ名
    f"--output_dir={OUTPUT_DIR}",
    f"--job_name=smolvla_libero_plus_{RUN_NAME}",
    # 学習ステップ / batch / DataLoader
    f"--steps={STEPS}",
    f"--batch_size={BATCH_SIZE}",
    "--num_workers=0",            # subprocess 内での追加 worker は無効化
    "--persistent_workers=false",
    # 学習中の env 評価は無効（このノート側で別途評価する）
    "--env_eval_freq=0",
    "--eval_steps=0",
    f"--seed={SEED}",
    # checkpoint は最終 step のみローカル保存
    "--save_checkpoint=true",
    f"--save_freq={STEPS}",
    "--save_checkpoint_to_hub=false",
    f"--log_freq={LOG_FREQ}",
    "--wandb.enable=false",
    # LoRA 設定
    "--peft.method_type=LORA",
    f"--peft.r={LORA_R}",
    f"--peft.lora_alpha={LORA_ALPHA}",
]

# --- 子プロセスの環境変数 ---------------------------------------
training_env = os.environ.copy()
# clone した src を PYTHONPATH の先頭に入れてパッチ済み LeRobot を使わせる
training_env["PYTHONPATH"] = (
    str(LEROBOT_SRC)
    + os.pathsep
    + training_env.get("PYTHONPATH", "")
)
# accelerate に混合精度モード（bf16/fp16）を伝える
training_env["ACCELERATE_MIXED_PRECISION"] = (
    MIXED_PRECISION
)
# stdout を行バッファ化して進捗をリアルタイム表示できるようにする
training_env["PYTHONUNBUFFERED"] = "1"
training_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
training_env["HF_HUB_VERBOSITY"] = "error"
training_env["TQDM_DISABLE"] = "1"
training_env["PYTHONWARNINGS"] = "ignore"
if RUN.get("blur"):
    assert os.path.exists("/content/train_blur.py"), \
        "6.1 の道具作成セルを先に実行してください"
    training_env["BLUR_P"] = str(RUN.get("blur_p", 0.5))
    if os.path.exists("/content/train_blur_log.txt"):
        os.remove("/content/train_blur_log.txt")

# 出力ディレクトリはクリーンな状態から始める
shutil.rmtree(OUTPUT_DIR, ignore_errors=True)

print("Preparing data and starting training...")

# --- サブプロセスを起動して stdout を逐次読み --------------------
process = subprocess.Popen(
    command,
    cwd=LEROBOT_DIR,
    env=training_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# 直近ログ 80 行を保持しておき、失敗時にダンプする（診断用）
recent_lines: deque[str] = deque(maxlen=80)
report_step = LOG_FREQ

assert process.stdout is not None

# "step:... loss:... lr:..." 行だけを拾って整形表示する
for raw_line in process.stdout:
    line = raw_line.replace("\r", "").strip()

    if not line:
        continue

    recent_lines.append(line)

    if "step:" in line and "loss:" in line:
        loss_match = re.search(r"loss:([0-9.eE+-]+)", line)
        lr_match = re.search(r"lr:([0-9.eE+-]+)", line)

        loss = loss_match.group(1) if loss_match else "n/a"
        lr = lr_match.group(1) if lr_match else "n/a"

        print(
            f"step {report_step:4d}/{STEPS}  "
            f"loss={loss}  lr={lr}"
        )
        report_step += LOG_FREQ

return_code = process.wait()

# 失敗時は末尾ログを吐いてから raise（原因調査用）
if return_code != 0:
    print("\n".join(recent_lines))
    raise RuntimeError(
        f"Training failed: {return_code}"
    )

print("Training complete.")

# 4周目: ぼやけがかかっていたことの確認(どのカメラに何回かかったか)
if RUN.get("blur") and os.path.exists("/content/train_blur_log.txt"):
    print(open("/content/train_blur_log.txt").read())

### 8.2b 学習直後にアダプタを Drive へ退避

マージ前の LoRA アダプタは小さい(数十 MB 程度)ので、学習が終わったらすぐ Drive に
コピーします。Colab が切れても、**再学習せずに 8.3 から再開**できます。
マージ済みの重み(約 1GB 超)を Drive に置くより容量を使いません。

In [ ]:
# 【役割】学習直後に、LoRA アダプタ(小さい)を Drive へ退避する
#   切れても再学習せず 8.3 から再開できる
import shutil

_ckpt = (OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}"
         / "pretrained_model")
assert (_ckpt / "adapter_model.safetensors").is_file(), "adapter なし"

_dst = DRIVE_RUN_DIR / "adapter" / "pretrained_model"
shutil.rmtree(_dst, ignore_errors=True)
shutil.copytree(_ckpt, _dst)

_mb = sum(p.stat().st_size for p in _dst.rglob("*")
          if p.is_file()) / 1e6
print(f"saved: {_dst}  ({_mb:.0f} MB)")

# 学習ログの要約(README / 説明資料用)も残す
import json
(DRIVE_RUN_DIR / "run_config.json").write_text(
    json.dumps({"run": RUN_NAME, **RUN, "seed": SEED,
                "episodes": len(EPISODE_INDICES)},
               ensure_ascii=False, indent=1),
    encoding="utf-8")

#### 8.2c 切断後の復元(自動)

ランタイムが切れたら、Section 1〜8.1 を実行し直したうえで、このセルを実行します。
**このセルは、学習結果が手元になく、Drive にアダプタがあるときだけ**、
アダプタを学習の出力先へ戻します(それ以外は何もしません)。
復元したら 8.2 と 8.2b は**飛ばして** 8.3 へ進んでください。

In [ ]:
# 【役割】切断後に、Drive のアダプタを学習の出力先へ戻す(自動判定)
import shutil

_dst = (OUTPUT_DIR / "checkpoints" / f"{STEPS:06d}" / "pretrained_model")
_src = DRIVE_RUN_DIR / "adapter" / "pretrained_model"

if (_dst / "adapter_model.safetensors").is_file():
    print("skip: 学習結果がすでにあります →", _dst)
elif (_src / "adapter_model.safetensors").is_file():
    shutil.rmtree(_dst, ignore_errors=True)
    _dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(_src, _dst)
    print("restored (8.2・8.2b は飛ばして 8.3 へ):", _dst)
else:
    print(f"Drive にアダプタがありません ({RUN_NAME})。8.2 で学習してください。")

#### (任意)1周目の重みを旧 zip から復元する

過去に Drive へ置いた 1周目のマージ済み重み(zip)があれば、1時間の学習を省けます。
**復元したら 8.2〜8.3 は飛ばして、8.4 に進んでください。**
`RUN_NAME = "r1_spatial"` のときだけ使えます。

In [ ]:
# 【役割】1周目のマージ済み重み(Drive の zip)を復元する
#   使うなら 8.2〜8.3 は飛ばして 8.4 へ
RESTORE_LEGACY_R1 = False   # 使うときだけ True

if RESTORE_LEGACY_R1:
    import shutil
    assert RUN_NAME == "r1_spatial", "1周目専用"
    _cands = [
        DRIVE_BACKUP_DIR / "smolvla_libero_plus_spatial_lora_merged.zip",
        DRIVE_BACKUP_DIR / "merged_r1_backup.zip",
    ]
    _zip = next(p for p in _cands if p.is_file())
    _tmp = WORKDIR / "_r1_unpack"
    shutil.rmtree(_tmp, ignore_errors=True)
    shutil.unpack_archive(_zip, _tmp)
    _inner = next(p for p in _tmp.iterdir() if p.is_dir())
    shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
    shutil.move(str(_inner), MERGED_MODEL_DIR)
    print("restored:", MERGED_MODEL_DIR)
    print(sorted(p.name for p in MERGED_MODEL_DIR.iterdir()))
else:
    print("skip (RESTORE_LEGACY_R1=False)")

### 8.3 LoRA をベースへマージ

LoRA差分を元weightへ統合し、通常のLeRobotモデルとして保存します。

In [ ]:
# ==============================================================
# 【役割】LoRA の差分をベースの重みに足し込み、単体モデルとして保存
#   評価のとき peft を介さずに済む。前処理・後処理の統計量も一緒にコピー
# 【出力】MERGED_MODEL_DIR (追加学習後モデル)
# ==============================================================
# 学習後の adapter を PeftModel でロード → merge_and_unload で
# LoRA 差分をベース weight に足し込み、通常の LeRobot モデル形式で保存する。
# こうすることで評価時に peft を介さず素の推論経路で動作する。
import contextlib
import gc
import io
import json

from peft import PeftModel
from safetensors import safe_open
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

# lerobot-train が保存した最終 checkpoint（adapter が入っている）
checkpoint_dir = (
    OUTPUT_DIR
    / "checkpoints"
    / f"{STEPS:06d}"
    / "pretrained_model"
)

if not (
    checkpoint_dir / "adapter_model.safetensors"
).is_file():
    raise FileNotFoundError("Final adapter not found.")

# GPU メモリを空けてから CPU 上でマージ（VRAM を節約）
gc.collect()
torch.cuda.empty_cache()

# --- ベース + LoRA のロード設定 ---------------------------------
merge_config = PreTrainedConfig.from_pretrained(
    checkpoint_dir
)
merge_config.device = "cpu"                     # マージは CPU で実行
merge_config.pretrained_path = BASE_MODEL_LOCAL # ベース weight の場所を明示
merge_config.use_peft = False                   # マージ後は peft 経路を使わない

# ロード時の noisy な info ログを黙らせる
quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 1) ベース SmolVLA を CPU にロード
    base_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=merge_config,
        strict=False,
    )

    # 2) 学習済み adapter を被せる
    peft_policy = PeftModel.from_pretrained(
        base_policy,
        checkpoint_dir,
        is_trainable=False,
        torch_device="cpu",
    )

    # 3) LoRA を base に足し込んで単体モデル化
    merged_policy = peft_policy.merge_and_unload(
        safe_merge=True
    )

# --- マージ済みモデルを保存 -------------------------------------
shutil.rmtree(MERGED_MODEL_DIR, ignore_errors=True)
MERGED_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config は「単体モデル・再 push 無し・VLM は再ロードしない」に固定
merged_policy.config.use_peft = False
merged_policy.config.pretrained_path = None
merged_policy.config.push_to_hub = False
merged_policy.config.repo_id = None
merged_policy.config.device = None
merged_policy.config.load_vlm_weights = False
merged_policy.config.vlm_model_name = VLM_REPO

merged_policy.save_pretrained(MERGED_MODEL_DIR)

# 学習中に更新された前処理/後処理 stats も同じディレクトリへコピーしないと
# 評価時に入出力スケーリングが崩れる
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in checkpoint_dir.glob(pattern):
        shutil.copy2(
            source_path,
            MERGED_MODEL_DIR / source_path.name,
        )

# --- マージ結果に LoRA パラメータが残っていないか検証 ------------
merged_weights_path = (
    MERGED_MODEL_DIR / "model.safetensors"
)

with safe_open(
    merged_weights_path,
    framework="pt",
    device="cpu",
) as weights:
    if any(
        "lora_" in key.lower()
        for key in weights.keys()
    ):
        raise RuntimeError(
            "LoRA parameters remain after merge."
        )

# CPU 上に膨らんだモデルを片付ける（次の baseline ロードが重い）
del peft_policy
del base_policy
del merged_policy

gc.collect()
torch.cuda.empty_cache()

print("Merged model ready.")

### 8.4 対照ベースラインを準備

公開weightを追加学習モデルと同じ入力schema・processorへ揃えます。

In [ ]:
# 【役割】8.3 を飛ばして来ても動くよう、必要な import をここにも置く
import contextlib
import gc
import io
import json
import shutil

import torch
from lerobot.configs import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import (
    SmolVLAPolicy,
)

# ==============================================================
# 比較用ベースラインを追加学習モデルと同じ schema で保存し直す
# ==============================================================
# 公開の初期重みをそのまま評価するのではなく、マージ済みモデルと同じ
# processor 設定・入出力 features に揃えた「対照モデル」として保存し直す。
# これで評価時の前処理/後処理を完全に一致させ、比較を公正にする。

# マージ済みモデルの config を土台にする（入出力 schema が同一になる）
baseline_config = PreTrainedConfig.from_pretrained(
    MERGED_MODEL_DIR
)
baseline_config.device = "cpu"
baseline_config.pretrained_path = BASE_MODEL_LOCAL
baseline_config.use_peft = False
baseline_config.load_vlm_weights = False

quiet_output = io.StringIO()

with (
    contextlib.redirect_stdout(quiet_output),
    contextlib.redirect_stderr(quiet_output),
):
    # 公開の初期 weight を読みつつ、config だけマージ済み側と揃える
    baseline_policy = SmolVLAPolicy.from_pretrained(
        BASE_MODEL_LOCAL,
        config=baseline_config,
        strict=False,
    )

shutil.rmtree(BASELINE_MODEL_DIR, ignore_errors=True)
BASELINE_MODEL_DIR.mkdir(parents=True, exist_ok=True)

# 保存 config を単体モデル運用向けに固定（マージ済みモデルと同じ扱い）
baseline_policy.config.use_peft = False
baseline_policy.config.pretrained_path = None
baseline_policy.config.push_to_hub = False
baseline_policy.config.repo_id = None
baseline_policy.config.device = None
baseline_policy.config.load_vlm_weights = False
baseline_policy.config.vlm_model_name = VLM_REPO
baseline_policy.save_pretrained(BASELINE_MODEL_DIR)

# processor stats はマージ済み側からコピー（両者を完全一致させる）
for pattern in [
    "policy_preprocessor.json",
    "policy_preprocessor*.safetensors",
    "policy_postprocessor.json",
    "policy_postprocessor*.safetensors",
]:
    for source_path in MERGED_MODEL_DIR.glob(pattern):
        shutil.copy2(
            source_path,
            BASELINE_MODEL_DIR / source_path.name,
        )

del baseline_policy
gc.collect()
torch.cuda.empty_cache()

print("Baseline ready.")

### 8.5 Spatial の維持を確認する(3 タスク)

追加学習で、もともとできていた Spatial が壊れていないかを確認します。
同じタスク・同じ seed で、ベースライン と 追加学習後を評価します
(3 環境 × 1 エピソード × 2 モデル = 6 rollout)。

In [ ]:
# ==============================================================
# 【役割】追加学習前後のモデルを、同じタスク・同じ seed で評価する
#   Spatial の維持確認に使う (3 環境 × 1 エピソード)
# 【出力】BASE_EVAL_INFO / FINETUNED_EVAL_INFO (eval_info.json の内容)
# ==============================================================
# lerobot-eval を subprocess で 2 回実行し、Section 5 で仕込んだ
# "EVAL_PROGRESS ..." 行をパースしてリアルタイム進捗を表示する。
import json
import re
from collections import deque

# lerobot-eval CLI 引数を組み立てる。ベース/マージ済みで共通の設定。
def build_eval_command(
    policy_path: Path,
    output_dir: Path,
) -> list[str]:
    return [
        "lerobot-eval",
        f"--policy.path={policy_path}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        "--env.task=libero_spatial",
        (
            "--env.task_ids="
            + json.dumps(
                EVAL_TASK_IDS,
                separators=(",", ":"),
            )
        ),
        (
            "--env.camera_name_mapping="
            + json.dumps(
                EVAL_CAMERA_MAPPING,
                separators=(",", ":"),
            )
        ),
        # 観測画像は 256x256（SmolVLA の入力仕様に合わせる）
        "--env.observation_height=256",
        "--env.observation_width=256",
        # relative action 空間（LIBERO-plus のデフォルト）
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",  # 逐次実行（VRAM 節約）
        "--eval.batch_size=1",
        (
            "--eval.n_episodes="
            f"{EVAL_EPISODES_PER_TASK}"
        ),
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={output_dir}",
    ]


# 1 モデル分の評価を回して eval_info.json を dict で返す。
def run_evaluation(
    policy_path: Path,
    output_dir: Path,
    label: str,
) -> dict:
    shutil.rmtree(
        output_dir,
        ignore_errors=True,
    )

    # 評価用の子プロセスへ渡す環境変数
    eval_env = os.environ.copy()
    eval_env["MUJOCO_GL"] = "egl"
    # LIBERO-plus fork とパッチ済み LeRobot src の両方を優先パスに入れる
    eval_env["PYTHONPATH"] = (
        str(LIBERO_PLUS_DIR)
        + os.pathsep
        + str(LEROBOT_SRC)
        + os.pathsep
        + eval_env.get("PYTHONPATH", "")
    )
    eval_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"
    eval_env["HF_HUB_VERBOSITY"] = "error"
    eval_env["TQDM_DISABLE"] = "1"
    eval_env["PYTHONWARNINGS"] = "ignore"
    eval_env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        build_eval_command(policy_path, output_dir),
        cwd=LEROBOT_DIR,
        env=eval_env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )

    # 失敗時ダンプ用の直近ログ
    recent_lines: deque[str] = deque(maxlen=120)

    # Section 5 で仕込んだ "EVAL_PROGRESS task=x/y episode=a/b" 行を拾う正規表現
    progress_pattern = re.compile(
        r"^EVAL_PROGRESS "
        r"task=(\d+)/(\d+) "
        r"episode=(\d+)/(\d+)$"
    )

    # tqdm で全 rollout の進捗を出す
    from tqdm.auto import tqdm
    _total_rollouts = (
        len(EVAL_TASK_IDS) * EVAL_EPISODES_PER_TASK
    )
    _pbar_eval = tqdm(total=_total_rollouts, desc=f"eval[{label}]", unit="ep")

    assert process.stdout is not None
    for raw_line in process.stdout:
        line = raw_line.replace("\r", "").strip()

        if not line:
            continue

        recent_lines.append(line)
        match = progress_pattern.match(line)

        if match:
            ti, tt, ei, et = match.groups()
            _pbar_eval.update(1)
            _pbar_eval.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")

    _pbar_eval.close()
    return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            "\n".join(recent_lines)
        )

    # 評価結果 JSON を読んで返す
    result_path = (
        output_dir
        / "eval_info.json"
    )

    if not result_path.is_file():
        raise FileNotFoundError(
            result_path
        )

    return json.loads(
        result_path.read_text(
            encoding="utf-8"
        )
    )


# 追加学習前の（processor を揃えた）ベースラインを評価
BASE_EVAL_INFO = run_evaluation(
    BASELINE_MODEL_DIR,
    BASE_EVAL_DIR,
    "Base model",
)

# LoRA マージ済みモデルを同じ seed・同じ tasks で評価
FINETUNED_EVAL_INFO = run_evaluation(
    MERGED_MODEL_DIR,
    FINETUNED_EVAL_DIR,
    RUN_NAME,
)

print("Evaluation complete.")

### 8.6 LeRobot 版の成功率比較

`Δ (pp)`は、追加学習後から追加学習前を引いた成功率差です。

In [ ]:
# ==============================================================
# 【役割】タスクごとの成功率と、追加学習前後の差 (pp) を表にして CSV 保存
# 【出力】comparison_df / COMPARISON_CSV_PATH
# ==============================================================
import pandas as pd
from IPython.display import display


# eval_info.json の per_task から task_id → 成功率(%) を作る。
def per_task_success(
    eval_info: dict,
) -> dict[int, float]:
    result: dict[int, float] = {}

    for task_info in eval_info["per_task"]:
        task_id = int(task_info["task_id"])
        successes = task_info["metrics"]["successes"]
        # bool リスト → True の割合を %
        result[task_id] = (
            100.0
            * sum(bool(value) for value in successes)
            / len(successes)
        )

    return result


# task_id は「指示文 × 環境」の通し番号。実際の指示文を環境から引く。
def spatial_label(task_id: int) -> str:
    import contextlib, io
    try:
        from libero.libero import benchmark
        with contextlib.redirect_stdout(io.StringIO()):
            suite = benchmark.get_benchmark_dict()["libero_spatial"]()
        return suite.get_task(task_id).language
    except Exception:
        return f"task_id {task_id}"


base_per_task = per_task_success(BASE_EVAL_INFO)
finetuned_per_task = per_task_success(
    FINETUNED_EVAL_INFO
)

# --- 各 task 行を構築 -----------------------------------------
rows = []

for task_id in EVAL_TASK_IDS:
    base_score = base_per_task[task_id]
    finetuned_score = finetuned_per_task[task_id]

    rows.append(
        {
            "Task ID": task_id,
            "Task": spatial_label(task_id),
            "Base (%)": base_score,
            FT_COL: finetuned_score,
            # Δ (pp) は「追加学習後 − 追加学習前」のパーセンテージポイント差
            "Δ (pp)": finetuned_score - base_score,
        }
    )

# --- Overall 行（LIBERO-Spatial 全体の pc_success） ------------
base_overall = float(
    BASE_EVAL_INFO["overall"]["pc_success"]
)
finetuned_overall = float(
    FINETUNED_EVAL_INFO["overall"]["pc_success"]
)

rows.append(
    {
        "Task ID": "Overall",
        "Task": "LIBERO-Spatial",
        "Base (%)": base_overall,
        FT_COL: finetuned_overall,
        "Δ (pp)": finetuned_overall - base_overall,
    }
)

# DataFrame 化して CSV に保存し、画面にも表示
comparison_df = pd.DataFrame(rows)
comparison_df.to_csv(
    COMPARISON_CSV_PATH,
    index=False,
)

display(comparison_df.round(1))

print(
    f"Overall: {base_overall:.1f}% → "
    f"{finetuned_overall:.1f}% "
    f"({finetuned_overall - base_overall:+.1f} pp)"
)

### 8.7 評価結果を Drive に保存

Spatial の維持確認の結果を Drive に残します。
重みは保存せず、アダプタ(8.2b)から再生成します(容量の節約)。

In [ ]:
# 【役割】Spatial 維持確認の結果を Drive に保存する
import shutil

for _src, _name in [(BASE_EVAL_DIR, "spatial_base"),
                    (FINETUNED_EVAL_DIR, f"spatial_{RUN_NAME}")]:
    if _src.exists():
        shutil.copytree(_src, DRIVE_EVAL_DIR / _name,
                        dirs_exist_ok=True)
shutil.copy2(COMPARISON_CSV_PATH, DRIVE_RUN_DIR)
print("saved to", DRIVE_RUN_DIR)

## 9. 4 suite の評価

追加学習前後のモデルを、4 つの suite で評価します。

- 対象: `libero_spatial` / `libero_object` / `libero_goal` / `libero_10`
- 各 suite から task_id 0・4・8 の 3 環境 × 1 エピソード = 12 rollout / モデル
- **注意:** LIBERO-plus の `task_id` は、同じ指示文の環境違い(`table 1`、`tb 10` など)が大量に並んでいます。
  0・4・8 は『同じタスクの3つの環境』で、3種類のタスクではありません。
- 評価 seed は固定(同じ条件で比べるため)

入力: `BASELINE_MODEL_DIR`(追加学習前)と `MERGED_MODEL_DIR`(追加学習後)。
どちらも `model.safetensors`、`config.json`、前処理・後処理の設定と統計量が
そろっている必要があります(9.0 で確認します)。

出力(`eval/advanced/`):
- `<モデル>/<suite>/eval_info.json` … 各エピソードの成否
- `advanced_comparison.csv` … suite ごとの成功率の比較表

評価は 1 タスク 1 エピソードと小さいので、成功率はばらつきます。
差が出ても、有意な差とは言えません。

In [ ]:
# ==============================================================
# 【役割】評価に渡すモデルフォルダに、必要なファイルがそろっているか確認
#   足りなければここで止まる (評価の途中で落ちるのを防ぐ)
# ==============================================================

_REQUIRED_FILES = [
    "model.safetensors",
    "config.json",
    "policy_preprocessor.json",
    "policy_postprocessor.json",
]
# .safetensors の stats は名前が可変なので glob で検証
_REQUIRED_GLOBS = [
    "policy_preprocessor*.safetensors",
    "policy_postprocessor*.safetensors",
]

def _check_policy_dir(policy_dir: Path, label: str) -> None:
    print(f"\n[{label}] {policy_dir}")
    if not policy_dir.is_dir():
        raise FileNotFoundError(
            f"{label}: ディレクトリが存在しません。"
            "section 8 の該当セルを実行しましたか?"
        )
    missing = []
    for name in _REQUIRED_FILES:
        p = policy_dir / name
        ok = p.is_file()
        print(f"  {'✅' if ok else '❌'}  {name}"
              + (f"  ({p.stat().st_size / 1e6:.1f} MB)" if ok else ""))
        if not ok:
            missing.append(name)
    for pattern in _REQUIRED_GLOBS:
        matches = list(policy_dir.glob(pattern))
        ok = len(matches) > 0
        print(f"  {'✅' if ok else '❌'}  {pattern}  ({len(matches)} match)")
        if not ok:
            missing.append(pattern)
    if missing:
        raise FileNotFoundError(
            f"{label} に必要ファイルが不足: {missing}. "
            "section 8.3 (merge) / 8.4 (baseline) を再実行してください。"
        )


_check_policy_dir(BASELINE_MODEL_DIR, "BASELINE_MODEL_DIR (対照)")
_check_policy_dir(MERGED_MODEL_DIR, "MERGED_MODEL_DIR (追加学習後)")
print("\n✅ モデルフォルダ OK — 評価を実行できます")

In [ ]:
# ==============================================================
# アドバンスド評価: 4 suite × 3 環境 × 1 ep = 12 rollout / モデル
# ==============================================================
# 設計:
#  - suite ごとに結果を Drive へ保存し、済んだ suite は再実行しない
#    (切断してもやり直しが最小になる)
#  - ベースラインの結果は run をまたいで共有する(同じ重み・同じ seed)
#  - 過去の Drive キャッシュ(adv_eval_r2/baseline など)があれば再利用
import json
import re
import shutil
import subprocess
from collections import deque

import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

_ADV_TASK_IDS = [0, 4, 8]
ADVANCED_SUITES = {
    "libero_spatial": _ADV_TASK_IDS,
    "libero_object":  _ADV_TASK_IDS,
    "libero_goal":    _ADV_TASK_IDS,
    "libero_10":      _ADV_TASK_IDS,
}
ADVANCED_EPISODES_PER_TASK = 1
ADVANCED_ROOT = WORKDIR / "eval" / "advanced"
BASE_ADV_DIR = ADVANCED_ROOT / "baseline"
ADVANCED_OUT_DIR = ADVANCED_ROOT / RUN_NAME

REUSE_LEGACY_BASELINE = True
LEGACY_BASELINE_DIRS = [
    DRIVE_BACKUP_DIR / "adv_eval_r2" / "baseline",
    DRIVE_BACKUP_DIR / "adv_eval_r1" / "baseline",
]


def _seed_from(src_root: Path, dst_root: Path) -> None:
    """src_root/<suite>/eval_info.json があれば dst_root へ複写。"""
    for suite in ADVANCED_SUITES:
        s = src_root / suite
        d = dst_root / suite
        if (s / "eval_info.json").is_file() and not (
            d / "eval_info.json"
        ).is_file():
            shutil.copytree(s, d, dirs_exist_ok=True)


def run_advanced_eval(policy_path, out_root, label, drive_root):
    """1 モデルについて全 suite を回し suite→eval_info を返す。"""
    out_root.mkdir(parents=True, exist_ok=True)
    _seed_from(drive_root, out_root)         # Drive にあれば復元
    suite_results = {}
    for suite_name, task_ids in ADVANCED_SUITES.items():
        sub_dir = out_root / suite_name
        done = sub_dir / "eval_info.json"
        if done.is_file():
            print(f"[{label}] {suite_name}: 済み(再実行しない)")
            suite_results[suite_name] = json.loads(
                done.read_text(encoding="utf-8"))
            continue
        shutil.rmtree(sub_dir, ignore_errors=True)

        adv_env = os.environ.copy()
        adv_env["MUJOCO_GL"] = "egl"
        adv_env["PYTHONPATH"] = (
            str(LIBERO_PLUS_DIR) + os.pathsep + str(LEROBOT_SRC)
            + os.pathsep + adv_env.get("PYTHONPATH", "")
        )
        adv_env["LEROBOT_MAX_RECORDED"] = "0"
        adv_env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
        adv_env["TQDM_DISABLE"] = "1"
        adv_env["PYTHONWARNINGS"] = "ignore"
        adv_env["PYTHONUNBUFFERED"] = "1"

        adv_command = [
            "lerobot-eval",
            f"--policy.path={policy_path}",
            "--policy.device=cuda",
            "--policy.use_amp=false",
            "--env.type=libero",
            "--env.is_libero_plus=true",
            f"--env.task={suite_name}",
            "--env.task_ids="
            + json.dumps(task_ids, separators=(",", ":")),
            "--env.camera_name_mapping="
            + json.dumps(EVAL_CAMERA_MAPPING,
                         separators=(",", ":")),
            "--env.observation_height=256",
            "--env.observation_width=256",
            "--env.control_mode=relative",
            "--env.max_parallel_tasks=1",
            "--eval.batch_size=1",
            f"--eval.n_episodes={ADVANCED_EPISODES_PER_TASK}",
            "--eval.use_async_envs=false",
            "--eval.recording=false",
            f"--seed={EVAL_SEED}",
            f"--output_dir={sub_dir}",
        ]
        print(f"\n[{label}] {suite_name} "
              f"({len(task_ids)} tasks × "
              f"{ADVANCED_EPISODES_PER_TASK} ep)", flush=True)
        proc = subprocess.Popen(
            adv_command, cwd=LEROBOT_DIR, env=adv_env, text=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            bufsize=1,
        )
        assert proc.stdout is not None
        adv_recent: deque[str] = deque(maxlen=120)
        _prog = re.compile(
            r"^EVAL_PROGRESS task=(\d+)/(\d+) episode=(\d+)/(\d+)$")
        pbar = tqdm(total=len(task_ids) * ADVANCED_EPISODES_PER_TASK,
                    desc=f"{label} {suite_name}", unit="ep")
        for raw in proc.stdout:
            line = raw.replace("\r", "").strip()
            if not line:
                continue
            adv_recent.append(line)
            m = _prog.match(line)
            if m:
                ti, tt, ei, et = m.groups()
                pbar.update(1)
                pbar.set_postfix_str(f"task {ti}/{tt} ep {ei}/{et}")
        pbar.close()
        if proc.wait() != 0:
            raise RuntimeError("\n".join(adv_recent))
        info = json.loads(done.read_text(encoding="utf-8"))
        suite_results[suite_name] = info
        # suite ごとに Drive へ退避
        shutil.copytree(sub_dir, drive_root / suite_name,
                        dirs_exist_ok=True)
    return suite_results


# --- ベースライン(run 共通)-----------------------------------
if REUSE_LEGACY_BASELINE:
    for _legacy in LEGACY_BASELINE_DIRS:
        if _legacy.is_dir():
            _seed_from(_legacy, DRIVE_EVAL_DIR / "advanced_baseline")
            print("既存のベースライン結果を再利用:", _legacy)
print("=== Advanced eval: baseline ===")
adv_base = run_advanced_eval(
    BASELINE_MODEL_DIR, BASE_ADV_DIR, "Base",
    DRIVE_EVAL_DIR / "advanced_baseline",
)

print(f"\n=== Advanced eval: {RUN_NAME} ===")
adv_ft = run_advanced_eval(
    MERGED_MODEL_DIR, ADVANCED_OUT_DIR / "finetuned", RUN_NAME,
    DRIVE_EVAL_DIR / f"advanced_{RUN_NAME}",
)


def overall_across_suites(results):
    total_success, total_rollouts = 0.0, 0
    for info in results.values():
        for task in info["per_task"]:
            s = task["metrics"]["successes"]
            total_success += sum(bool(v) for v in s)
            total_rollouts += len(s)
    return (100.0 * total_success / total_rollouts
            if total_rollouts else 0.0)


adv_rows = []
for suite_name in ADVANCED_SUITES:
    b = adv_base[suite_name]["overall"]["pc_success"]
    f = adv_ft[suite_name]["overall"]["pc_success"]
    adv_rows.append({"Suite": suite_name,
                     "Tasks": len(ADVANCED_SUITES[suite_name]),
                     "Base (%)": b, FT_COL: f, "Δ (pp)": f - b})
bo, fo = overall_across_suites(adv_base), overall_across_suites(adv_ft)
adv_rows.append({"Suite": "Overall (12 rollouts)",
                 "Tasks": sum(len(v) for v in ADVANCED_SUITES.values()),
                 "Base (%)": bo, FT_COL: fo, "Δ (pp)": fo - bo})

adv_df = pd.DataFrame(adv_rows)
adv_csv = ADVANCED_OUT_DIR / "advanced_comparison.csv"
adv_df.to_csv(adv_csv, index=False)
shutil.copy2(adv_csv, DRIVE_RUN_DIR)
display(adv_df.round(1))
print(f"Overall (12 rollouts): {bo:.1f}% -> {fo:.1f}% ({fo - bo:+.1f} pp)")

## 10. Object 改善実験

1周目の結果(公開重みも含め、Object は 0/3)から、Object の失敗は
**データを足すだけでは直らない**可能性が高い、と考えています。

観察:
- ロボットは動いている(動作量は成功時と同程度)が、**物体に近づかず上部で動き回る**。
- 背景・照明・カメラ視点などが試行ごとに違う(LIBERO-plus の摂動)。
- 2周目(Object を学習に追加)でも学習 loss はほぼ平らだった。

ここでは **GPU の少ない順**に、周ごとに1つだけ変えて確認します。

1. 10.0 診断: 評価タスクの中身と、推論時の `n_action_steps`(一度に実行する動作の数)の既定値。
   **注意:** LIBERO-plus の `task_id` は、同じ指示文に環境の違い
   (`table 1`、`tb 10` など)が付いたものが大量に並んでいます。
   `task_id` 0〜28 は、**すべて alphabet soup** でした(別の物体ではない)。
   物体の種類ごとの `task_id` は、10.0 で指示文の一致から求めます。
2. 10.2 `n_action_steps` を変えて Object を評価・録画。
   小さくすると、見た景色に合わせてこまめに動作を計算し直す。
3. 10.3 結果を全ラウンドで比較。
4. 10.4 周ごとに何を変えたか、Object の成功数がどう変わったかの表を出す。

**2周目はここまでです(再学習なし)。** 3周目の学習(`r3_object`)は、
ここでの結果を見てから実行します。
3 を終えても成功が出ない場合は、**その事実と原因の考察を成果として提出**します。

In [ ]:
# ==============================================================
# 【役割】失敗の原因を、再学習なしで調べる
#   (a) モデルが一度に実行する動作数 n_action_steps の既定値を表示
#   (b) 評価に使うタスクの指示文が想定どおりか確認
# ==============================================================
import json

# (a) モデル設定: 一度の推論で何ステップ実行するか
for _name, _dir in [("baseline", BASELINE_MODEL_DIR),
                    (RUN_NAME, MERGED_MODEL_DIR)]:
    _cfg = json.loads((_dir / "config.json").read_text())
    print(f"{_name:20s} chunk_size={_cfg.get('chunk_size')}  "
          f"n_action_steps={_cfg.get('n_action_steps')}")

# (b) LIBERO-plus の task_id は「指示文 × 環境バリエーション」の通し番号。
#     同じ指示文に、' table 1' や ' tb 10' などの末尾が付いた別環境が
#     大量に並ぶ。Object の 10 種類の物体それぞれについて、
#     指示文が一致する最初の task_id を求める。
import contextlib, io
from libero.libero import benchmark


def first_ids(suite, base_names):
    """base_names の各指示文で始まる最初の task_id と件数を返す。"""
    with contextlib.redirect_stdout(io.StringIO()):
        b = benchmark.get_benchmark_dict()[suite]()
    firsts, counts = {}, {}
    for i in range(b.get_num_tasks()):
        lang = b.get_task(i).language.lower()
        for name in base_names:
            if lang.startswith(name):
                counts[name] = counts.get(name, 0) + 1
                firsts.setdefault(name, i)
    return firsts, counts, b.get_num_tasks()


try:
    _firsts, _counts, _total = first_ids("libero_object",
                                         OBJECT_TASK_NAMES)
    print(f"libero_object: 全 {_total} 件")
    for _n in OBJECT_TASK_NAMES:
        print(f"  id {_firsts[_n]:5d} から  ({_counts[_n]:4d} 件)  "
              f"{_n[12:50]}")
    OBJECT_REP_IDS = [_firsts[n] for n in OBJECT_TASK_NAMES]
    print("\nOBJECT_REP_IDS =", OBJECT_REP_IDS)
except Exception as e:  # fork の API 差に備える(診断なので止めない)
    print("benchmark の列挙に失敗:", type(e).__name__, e)

### 10.1 録画用パッチ(1 回だけ)

`lerobot_eval.py` の録画本数を環境変数 `LEROBOT_MAX_RECORDED` で切り替えられるようにします。
何度実行しても安全です。Section 5 のパッチが先に当たっている必要があります。

In [ ]:
# 【役割】評価スクリプトの録画本数を環境変数で切り替えられるようにする
#   (何度実行しても安全)
_p = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
_s = _p.read_text(encoding="utf-8")
_flex = ('max_episodes_rendered = int(__import__("os")'
         '.environ.get("LEROBOT_MAX_RECORDED", "0"))')
if _flex not in _s:
    _old = "max_episodes_rendered = 0"
    assert _old in _s, "Section 5 のパッチが未適用です"
    _p.write_text(_s.replace(_old, _flex, 1), encoding="utf-8")
print("patched:", _flex in _p.read_text(encoding="utf-8"))

### 10.2 `n_action_steps` を変えて Object を評価・録画

- `NAS_LIST` の `None` は「モデルの既定値のまま」。
- 評価する `task_id` は、10.0 で求めた **10 種類の物体それぞれの最初の環境**(`OBJECT_REP_IDS`)です。
- 1 つの組み合わせ(モデル × 設定)は 10 rollout、目安は約 10 分です。
  既定は 2 モデル × 2 設定 = 40 rollout(約 40 分)。上から順に実行され、
  途中で止めても、済んだ分は残ります。
- 成功・失敗は mp4 のファイル名(`..._OK.mp4` / `..._NG.mp4`)と表に出ます。
- 結果は Drive の `object_sweep.json` に追記され、**済んだ組み合わせは飛ばします**。

In [ ]:
# ==============================================================
# 【役割】2周目の実験。n_action_steps を変えて Object を評価・録画する
#   済んだ組み合わせは飛ばし、結果を Drive の object_sweep.json に追記
# 【出力】OK/NG 付きの mp4 (Drive の videos/)、成功数の表
# ==============================================================
import json, os, shutil, subprocess
import pandas as pd
from IPython.display import display

SWEEP_SUITE = "libero_object"
# 10 種類の物体それぞれの最初の環境 (10.0 で求める。なければ 0・4・8)
SWEEP_TASK_IDS = (OBJECT_REP_IDS if "OBJECT_REP_IDS" in globals()
                  else [0, 4, 8])
NAS_LIST = [None, 10]                # None=既定値。5, 25 なども試せる
SWEEP_MODELS = {                     # 上から順に実行する
    RUN_NAME: MERGED_MODEL_DIR,
    "base": BASELINE_MODEL_DIR,
}

VID_ROOT = WORKDIR / "eval" / "video_batch"
VID_DRIVE = DRIVE_BACKUP_DIR / "videos"
VID_DRIVE.mkdir(parents=True, exist_ok=True)
SWEEP_JSON = DRIVE_RUN_DIR / "object_sweep.json"
sweep_log = (json.loads(SWEEP_JSON.read_text())
             if SWEEP_JSON.is_file() else [])


def record_eval(label, model_dir, suite, tids, nas=None):
    """tids を 1 本ずつ評価し、mp4 を Drive へ保存。結果 dict のリスト。"""
    tag_nas = "default" if nas is None else f"nas{nas}"
    out = VID_ROOT / f"{label}_{tag_nas}_{suite}"
    shutil.rmtree(out, ignore_errors=True)
    env = os.environ.copy()
    env["MUJOCO_GL"] = "egl"
    env["PYTHONPATH"] = os.pathsep.join([
        str(LIBERO_PLUS_DIR), str(LEROBOT_SRC),
        env.get("PYTHONPATH", ""),
    ])
    env["LEROBOT_MAX_RECORDED"] = "1"
    env["TQDM_DISABLE"] = "1"
    env["PYTHONWARNINGS"] = "ignore"
    cam = json.dumps(EVAL_CAMERA_MAPPING, separators=(",", ":"))
    cmd = [
        "lerobot-eval",
        f"--policy.path={model_dir}",
        "--policy.device=cuda",
        "--policy.use_amp=false",
        "--env.type=libero",
        "--env.is_libero_plus=true",
        f"--env.task={suite}",
        "--env.task_ids=" + json.dumps(tids, separators=(",", ":")),
        f"--env.camera_name_mapping={cam}",
        "--env.observation_height=256",
        "--env.observation_width=256",
        "--env.control_mode=relative",
        "--env.max_parallel_tasks=1",
        "--eval.batch_size=1",
        "--eval.n_episodes=1",
        "--eval.use_async_envs=false",
        "--eval.recording=false",
        f"--seed={EVAL_SEED}",
        f"--output_dir={out}",
    ]
    if nas is not None:
        cmd.insert(5, f"--policy.n_action_steps={nas}")
    r = subprocess.run(cmd, cwd=LEROBOT_DIR, env=env,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:], r.stderr[-1500:])
        return []
    info = json.loads(next(out.rglob("eval_info.json")).read_text())
    rows = []
    for t in info["per_task"]:
        tid = t["task_id"]
        ok = bool(t["metrics"]["successes"][0])
        mp4s = sorted(out.rglob(f"videos/{suite}_{tid}/*.mp4"))
        if mp4s:
            name = (f"{label}_{tag_nas}_{suite}_t{tid}_"
                    f"{'OK' if ok else 'NG'}.mp4")
            shutil.copy(mp4s[0], VID_DRIVE / name)
        rows.append({"model": label, "nas": tag_nas, "suite": suite,
                     "task_id": tid, "success": ok})
        print(label, tag_nas, suite, tid, "OK" if ok else "NG",
              flush=True)
    return rows


for label, model_dir in SWEEP_MODELS.items():
    for nas in NAS_LIST:
        tag_nas = "default" if nas is None else f"nas{nas}"
        todo = [t for t in SWEEP_TASK_IDS
                if not any(r["model"] == label and r["nas"] == tag_nas
                           and r["suite"] == SWEEP_SUITE
                           and r["task_id"] == t for r in sweep_log)]
        if not todo:
            print("skip", label, tag_nas)
            continue
        sweep_log += record_eval(label, model_dir, SWEEP_SUITE,
                                 todo, nas)
        SWEEP_JSON.write_text(json.dumps(sweep_log, indent=1))

sw = pd.DataFrame(sweep_log)
# task_id → 物体名 (10.0 で求めたグループがあれば)
if "OBJECT_REP_IDS" in globals():
    _names = {
        t: n.replace("pick up the ", "").split(" and place")[0]
        for t, n in zip(OBJECT_REP_IDS, OBJECT_TASK_NAMES)
    }
    sw["task"] = sw["task_id"].map(
        lambda t: _names.get(t, f"alphabet soup (id {t})"))
else:
    sw["task"] = sw["task_id"].astype(str)
display(sw.pivot_table(index=["model", "nas"], columns="task",
                       values="success", aggfunc="first"))
print(sw.groupby(["model", "nas"])["success"].agg(["sum", "count"]))

### 10.3 全ラウンドの比較表

Drive に保存済みの結果から、ベースライン・1周目・2周目・今回の run を
タスク単位で並べます(存在しないものは飛ばします)。

In [ ]:
# ==============================================================
# 【役割】保存済みの評価結果を読み、周ごとのタスク別成功率を 1 つの表にする
# 【出力】task_breakdown_all_rounds.csv
# ==============================================================
import json
import pandas as pd

D = DRIVE_BACKUP_DIR
SOURCES = {   # 列名: その eval_info.json が入っているフォルダ
    "baseline":          [DRIVE_EVAL_DIR / "advanced_baseline",
                          D / "adv_eval_r2" / "baseline"],
    "r1_spatial":        [DRIVE_EVAL_DIR / "advanced_r1_spatial",
                          D / "adv_eval_r1" / "finetuned"],
    "r1b_spatial_object": [DRIVE_EVAL_DIR
                           / "advanced_r1b_spatial_object",
                           D / "adv_eval_r2" / "finetuned"],
    "r3_object":        [DRIVE_EVAL_DIR / "advanced_r3_object"],
}


def load_first(folders):
    for folder in folders:
        out = {}
        for suite in ADVANCED_SUITES:
            p = folder / suite / "eval_info.json"
            if not p.is_file():
                out = None
                break
            for t in json.loads(p.read_text())["per_task"]:
                ok = t["metrics"]["successes"]
                out[(suite, t["task_id"])] = float(sum(ok)) / len(ok)
        if out:
            return out
    return None


cols = {}
for name, folders in SOURCES.items():
    r = load_first(folders)
    if r is None:
        print("なし:", name)
    else:
        cols[name] = r
tbl = pd.DataFrame(cols)
tbl.index.names = ["suite", "task_id"]
print(tbl)
print("\nsuite 別 成功数 (/3):")
print(tbl.groupby("suite").sum())
tbl.to_csv(DRIVE_BACKUP_DIR / "task_breakdown_all_rounds.csv")

### 10.4 3周の変更点と Object の成功数

- 周ごとの変更点は、`PRESETS` と `NAS_LIST` から作ります。
- Object の成功数は、10.2 の保存結果(各 run の `object_sweep.json`)から集計します。
  1周目 = r1_spatial の既定値、2周目 = r1_spatial の n_action_steps 変更、
  3周目 = r3_object(採用した n_action_steps)です。
- 表は README と説明資料の「工夫した点」にそのまま使えます。

In [ ]:
# 【役割】3周の変更点と、Object の成功数を表にまとめる
# 【出力】changes_3rounds.csv / object_by_round.csv
import json
import pandas as pd
from IPython.display import display

# --- 周ごとの変更点(手で書く。変えるのは1周に1つ) ---------------
A, C = PRESETS["r1_spatial"], PRESETS["r3_object"]
diff = [f"{k}: {A[k]} → {C[k]}" for k in A if A[k] != C[k]]
if "eps_by_task" in C:
    diff.append(f"失敗した物体 {len(C['eps_by_task'])} 種類は各 "
                f"{max(C['eps_by_task'].values())} 本")
changes = pd.DataFrame([
    {"周": 1, "変えたもの": "基準", "中身": "Spatial のみ学習"},
    {"周": 2, "変えたもの": "推論の n_action_steps",
     "中身": "再学習なし。既定値 と 小さい値を比較"},
    {"周": 3, "変えたもの": "学習",
     "中身": "; ".join(diff)},
])
display(changes)
changes.to_csv(DRIVE_BACKUP_DIR / "changes_3rounds.csv", index=False)

# --- Object の成功数(10.2 の保存結果)-------------------------
rows = []
for run in ["r1_spatial", "r3_object"]:
    p = DRIVE_BACKUP_DIR / "runs" / run / "object_sweep.json"
    if not p.is_file():
        print("なし:", p)
        continue
    for r in json.loads(p.read_text()):
        rows.append({"run": run, **r})
if rows:
    sw = pd.DataFrame(rows)
    out = (sw.groupby(["run", "model", "nas"])["success"]
             .agg(成功="sum", 試行="count").reset_index())
    display(out)
    out.to_csv(DRIVE_BACKUP_DIR / "object_by_round.csv", index=False)

## 11. 見直し間隔の比較(推論の `n_action_steps` を5通り)

2周目で効いた「見直しの間隔」を、5つの設定で比べます。
**同じ30回(物体10種 × 環境3通り)**を、設定ごとに評価します。
実用面の指標として、1回の推論にかかる時間も計ります。

| 設定 | 内容 |
|---|---|
| fixed50 / 25 / 10 / 5 | 何動作ごとに見直すか(固定) |
| adaptive_10_25 | つかむ前は10動作、つかんでいる間は25動作ごと |

**準備:** 6.1 の道具作成と、8.3 のマージが済んでいること(`MERGED_MODEL_DIR`)。
所要: 約3時間(5設定 × 約36分)。切断しても、済んだ設定は `replan_sweep.json` から
読み直して飛ばします。

### 11.1 動作確認(alphabet soup の3環境 × 2設定)

In [ ]:
import sys, json, importlib
sys.path.insert(0, "/content")
import replan_sweep as rs
importlib.reload(rs)

# alphabet soup の3つの環境で、固定10 と 適応型 を 1 回ずつ試す
ids = rs.object_variant_ids(OBJECT_TASK_NAMES, per_obj=3)
tids = ids[OBJECT_TASK_NAMES[0]]
print("task_ids:", tids)

OUT = WORKDIR / "eval" / "replan_smoke"
kw = dict(
    tools_dir="/content", lerobot_dir=LEROBOT_DIR,
    lerobot_src=LEROBOT_SRC, libero_plus_dir=LIBERO_PLUS_DIR,
    camera_mapping=EVAL_CAMERA_MAPPING, seed=EVAL_SEED,
)
for st in (rs.SETTINGS[2], rs.SETTINGS[4]):
    row = rs.run_one(MERGED_MODEL_DIR, st, tids, OUT, **kw)
    if row is None:
        break
    print(st["name"], "成功", row["success"], "/", row["n"],
          "| 推論回数/エピソード", round(row["calls_per_episode"], 1),
          "| ms/回", round(row["ms_per_call"]),
          "| 実時間(s)", row["wall_s"])
    stats = json.loads((OUT / f"{st['name']}_stats.json").read_text())
    w = stats["episodes"][0]["w"]
    print("  グリッパー幅(10ステップごと):", w[::10])

### 11.2 5設定 × 30回の評価

In [ ]:
import sys, json, importlib
sys.path.insert(0, "/content")
import replan_sweep as rs
importlib.reload(rs)

ids = rs.object_variant_ids(OBJECT_TASK_NAMES, per_obj=3)
print({k[-30:]: v for k, v in ids.items()})

# 大事な設定から先に: 固定10, 適応型, 固定50, 固定25, 固定5
order = [rs.SETTINGS[i] for i in (2, 4, 0, 1, 3)]
kw = dict(
    tools_dir="/content", lerobot_dir=LEROBOT_DIR,
    lerobot_src=LEROBOT_SRC, libero_plus_dir=LIBERO_PLUS_DIR,
    camera_mapping=EVAL_CAMERA_MAPPING, seed=EVAL_SEED,
)
rows = rs.run_sweep(
    MERGED_MODEL_DIR, order, ids,
    out_root=WORKDIR / "eval" / "replan_sweep",
    save_json=DRIVE_RUN_DIR / "replan_sweep.json", **kw)
display(rs.summarize(rows))

## 12. 環境 A / B / C の確認(動画つき)

LIBERO-plus が、同じ指示文に付ける「環境の違い」を、3グループに分けて見ます。
分類は、LIBERO-plus の `task_classification.json` に従います
(**分類表の `id` = `task_id` + 1**)。

| グループ | 分類表の名前 | 見た目 | 回数 |
|---|---|---|---|
| A 背景の模様 | Background Textures | 背景・床の模様、色が変わる | 10 |
| B ぼやけ・カメラ | Sensor Noise(8)+ Camera Viewpoints(2) | メインカメラの映像が**ぼやける**(モーションブラー6・ガウスぼかし2)/ カメラの向きが変わる | 10 |
| C 照明 | Light Conditions | 明るさ・光の向きが変わる | 10 |

**注意:** 録画の映像は、ぼやけをかける前の映像です。ぼやけは、モデルに渡す画像にだけかかります。

### 12.1 録画用パッチ(10.1 を実行済みなら不要。何度実行しても安全)

In [ ]:
# 【役割】評価スクリプトの録画本数を環境変数で切り替えられるようにする
#   (何度実行しても安全)
_p = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
_s = _p.read_text(encoding="utf-8")
_flex = ('max_episodes_rendered = int(__import__("os")'
         '.environ.get("LEROBOT_MAX_RECORDED", "0"))')
if _flex not in _s:
    _old = "max_episodes_rendered = 0"
    assert _old in _s, "Section 5 のパッチが未適用です"
    _p.write_text(_s.replace(_old, _flex, 1), encoding="utf-8")
print("patched:", _flex in _p.read_text(encoding="utf-8"))

### 12.2 6本の動画を撮る(A・B・C × 2例)

In [ ]:
import os, sys, json, importlib, shutil
from pathlib import Path
os.environ["LEROBOT_MAX_RECORDED"] = "1"      # 1エピソードだけ録画
sys.path.insert(0, "/content")
import replan_sweep as rs
importlib.reload(rs)

kw = dict(tools_dir="/content", lerobot_dir=LEROBOT_DIR,
          lerobot_src=LEROBOT_SRC, libero_plus_dir=LIBERO_PLUS_DIR,
          camera_mapping=EVAL_CAMERA_MAPPING, seed=EVAL_SEED)
# (グループ, 物体, task_id, 内容)  ← 分類表(id = task_id + 1)で確認済み
CASES = [("A", "ketchup", 127, "背景の模様"),
         ("A", "alphabet_soup", 0, "背景の模様"),
         ("B", "alphabet_soup", 1410, "ぼやけ"),
         ("B", "ketchup", 820, "カメラの向き"),
         ("C", "ketchup", 2403, "照明"),
         ("C", "alphabet_soup", 2261, "照明")]
VID = Path("/content/cond_videos")
shutil.rmtree(VID, ignore_errors=True)
VID.mkdir()
log = []
for grp, obj, tid, what in CASES:
    st = dict(name=f"vid_{grp}_{obj}_{tid}", mode="fixed", nas=10)
    row = rs.run_one(MERGED_MODEL_DIR, st, [tid],
                     WORKDIR / "eval" / "vid", **kw)
    if row is None:
        print("失敗", grp, obj, tid)
        continue
    ok = "OK" if row["per_task"][tid] else "NG"
    mp4s = sorted((WORKDIR / "eval" / "vid" / st["name"]).rglob("*.mp4"))
    print(grp, what, obj, tid, ok, "動画", len(mp4s), "本")
    if mp4s:
        shutil.copy(mp4s[0], VID / f"cond{grp}_{obj}_t{tid}_{ok}.mp4")
    log.append([grp, what, obj, tid, ok, len(mp4s)])
(DRIVE_RUN_DIR / "cond_videos_log.json").write_text(
    json.dumps(log, ensure_ascii=False))
shutil.make_archive("/content/cond_videos", "zip", VID)
from google.colab import files
files.download("/content/cond_videos.zip")

### 12.3 条件別の成功数を出す関数

評価結果(`per_task`: task_id → 成功か)を、A / B / C に振り分けて数えます。
13 以降の結果にも使います。

In [ ]:
import collections, json
from pathlib import Path

_CAT_JSON = next(Path(LIBERO_PLUS_DIR).rglob("task_classification.json"), None)
_CAT = {}
if _CAT_JSON is not None:
    _CAT = {t["id"]: t["category"]
            for t in json.loads(_CAT_JSON.read_text())["libero_object"]}
_COND = {"Background Textures": "A 背景の模様",
         "Sensor Noise": "B ぼやけ",
         "Camera Viewpoints": "B カメラの向き",
         "Light Conditions": "C 照明"}


def cond_counts(per_task):
    """per_task: {task_id: 成功か}。分類表の id = task_id + 1。"""
    c = collections.defaultdict(lambda: [0, 0])
    for k, v in per_task.items():
        name = _COND.get(_CAT.get(int(k) + 1), "?")
        c[name][1] += 1
        c[name][0] += bool(v)
    out = {k: f"{a}/{b}" for k, (a, b) in sorted(c.items())}
    ab = [x for k, x in c.items() if k.startswith("B")]
    out["B 合計"] = f"{sum(a for a, _ in ab)}/{sum(b for _, b in ab)}"
    out["全体"] = (f"{sum(a for a, _ in c.values())}/"
                   f"{sum(b for _, b in c.values())}")
    return out


print("分類表:", _CAT_JSON, "| 件数", len(_CAT))
_sw = DRIVE_RUN_DIR / "replan_sweep.json"
if _sw.is_file():
    for r in json.loads(_sw.read_text()):
        print(f'{r["setting"]:16s}', cond_counts(r["per_task"]))
else:
    print("replan_sweep.json はまだありません(11.2 を実行すると出ます)")

## 13. ぼやけ対策①: 推論の前に画像を鮮明にする

B の8回は、**メインカメラの映像がぼやけた**状態です(手首カメラは、きれいなまま)。
モデルに渡す直前に、メインカメラの画像へ**アンシャープマスク**をかけます
(`x + 量 × (x − ぼかした x)` で輪郭を強調する)。学習はやり直しません。

- 対象のカメラ名は、`front` / `image` / `camera1`(`PRE_KEYS`)。
  **手首カメラ(`wrist`)にはかけません。**
- 補正が本当にかかったことは、ファイル `pre_mark.txt`(かかったカメラ名)で確認します。
  (かからないと、設定を変えても結果が1回も変わりません)
- 基準は、補正なしの 0/8(11.2 の `fixed10`)。

In [ ]:
import os, sys, json, importlib
sys.path.insert(0, "/content")
import replan_sweep as rs
importlib.reload(rs)
os.environ["PRE_MARK"] = "/content/pre_mark.txt"   # 補正がかかった印
if os.path.exists("/content/pre_mark.txt"):
    os.remove("/content/pre_mark.txt")

BLUR_IDS = [1410, 1443, 1484, 1524, 1614, 1662, 1742, 1782]  # ぼやけの8回
kw = dict(tools_dir="/content", lerobot_dir=LEROBOT_DIR,
          lerobot_src=LEROBOT_SRC, libero_plus_dir=LIBERO_PLUS_DIR,
          camera_mapping=EVAL_CAMERA_MAPPING, seed=EVAL_SEED)
OUT = WORKDIR / "eval" / "blur_test2"
SET = [
    dict(name="blur2_weak", mode="fixed", nas=10, pre="unsharp",
         pre_sigma=1.5, pre_amount=1.0),
    dict(name="blur2_strong", mode="fixed", nas=10, pre="unsharp",
         pre_sigma=3.0, pre_amount=2.0),
]
rows = []
for st in SET:
    row = rs.run_one(MERGED_MODEL_DIR, st, BLUR_IDS, OUT, **kw)
    if row is None:
        break
    rows.append(row)
    print(st["name"], "成功", row["success"], "/", row["n"],
          row["per_task"], flush=True)
(DRIVE_RUN_DIR / "blur_test2.json").write_text(
    json.dumps(rows, ensure_ascii=False, indent=1))
print("補正がかかったカメラ:",
      open("/content/pre_mark.txt").read()
      if os.path.exists("/content/pre_mark.txt") else "なし(かかっていません)")

## 14. ぼやけ対策②: 学習にぼやけを混ぜる(4周目 `r4_blur`)

3周目と**同じデータ・同じ学習設定**で、学習画像のメインカメラに「ぼやけ」を
確率 0.5 で混ぜて学習します(変えるのは、これだけ)。
ぼやけは、モーションブラー(7割、長さ 5〜15・向きはランダム)と、
ガウスぼかし(3割、σ=1〜3)です。手首カメラにはかけません。

**実行の手順**(切断後は、先に Section 1〜7 と 6.1 を実行):
1. Section 6 の `RUN_NAME` を **`"r4_blur"`** に書き換えて実行する
2. Section 7(学習データ選抜)→ 8.0 → 8.1
3. 8.2(学習。約1時間。ぼやけがかかっていないと、数分で止まります)→ 8.2b(Drive へ退避)
4. 8.3(マージ)
5. 下の 14.1(30回の評価と動画)

学習のあとに、`print` された `blurred_keys=[...]` に **`front`(または `image`)が出ること**を
確認します(出た名前のカメラにだけ、ぼやけがかかっています)。

### 14.1 4周目のモデルを評価(30回・補正なし・動画つき)

In [ ]:
import os, sys, json, importlib, shutil
from pathlib import Path
assert RUN_NAME == "r4_blur", "RUN_NAME を r4_blur にして 8.3 まで実行してください"
# 録画パッチ(12.1 と同じ。何度実行しても安全)
_p = LEROBOT_SRC / "lerobot" / "scripts" / "lerobot_eval.py"
_s = _p.read_text(encoding="utf-8")
_flex = ('max_episodes_rendered = int(__import__("os")'
         '.environ.get("LEROBOT_MAX_RECORDED", "0"))')
if _flex not in _s:
    assert "max_episodes_rendered = 0" in _s
    _p.write_text(_s.replace("max_episodes_rendered = 0", _flex, 1),
                  encoding="utf-8")
os.environ["LEROBOT_MAX_RECORDED"] = "1"
sys.path.insert(0, "/content")
import replan_sweep as rs
importlib.reload(rs)

kw = dict(tools_dir="/content", lerobot_dir=LEROBOT_DIR,
          lerobot_src=LEROBOT_SRC, libero_plus_dir=LIBERO_PLUS_DIR,
          camera_mapping=EVAL_CAMERA_MAPPING, seed=EVAL_SEED)
ids_by = rs.object_variant_ids(OBJECT_TASK_NAMES, per_obj=3)
ALL = [i for xs in ids_by.values() for i in xs]
OUT = WORKDIR / "eval" / "r4_all"
st = dict(name="r4_fixed10", mode="fixed", nas=10)
row = rs.run_one(MERGED_MODEL_DIR, st, ALL, OUT, **kw)
print("成功", row["success"], "/", row["n"], flush=True)
print(row["per_task"], flush=True)
(DRIVE_RUN_DIR / "r4_all.json").write_text(
    json.dumps(row, ensure_ascii=False, indent=1))

VID = Path("/content/r4_videos")
shutil.rmtree(VID, ignore_errors=True)
VID.mkdir()
for tid, ok in row["per_task"].items():
    for m in (OUT / st["name"]).rglob(f"*_{tid}/*.mp4"):
        shutil.copy(m, VID / f"t{tid}_{'OK' if ok else 'NG'}.mp4")
        break
shutil.make_archive("/content/r4_videos", "zip", VID)
from google.colab import files
files.download("/content/r4_videos.zip")

## 15. 結果のまとめ(周ごとの条件別の成功数)

Drive に保存した結果から、3周目と4周目を、**A / B / C 別**に並べます。
(12.3 の関数 `cond_counts` が必要です。どちらかの結果がなければ飛ばします。)

In [ ]:
import json
rows = []
for name, rel in [("3周目 r3_object", "r3_object/replan_sweep.json"),
                  ("4周目 r4_blur", "r4_blur/r4_all.json")]:
    p = DRIVE_BACKUP_DIR / "runs" / rel
    if not p.is_file():
        print("なし:", p)
        continue
    d = json.loads(p.read_text())
    if isinstance(d, list):                      # replan_sweep.json
        d = next(r for r in d if r["setting"] == "fixed10")
    rows.append((name, cond_counts(d["per_task"])))
keys = ["A 背景の模様", "B ぼやけ", "B カメラの向き", "B 合計",
        "C 照明", "全体"]
print(f'{"":18s}' + "".join(f"{k:>16s}" for k in keys))
for name, c in rows:
    print(f"{name:18s}" + "".join(f"{c.get(k, '-'):>16s}" for k in keys))